# InvariantRRF — canonical strict four-dataset rerun v4 (deep-dense + task-adaptive k)

**Purpose.** This notebook preserves the canonical leakage-clean, strict-checkpoint four-dataset experiment and adds three secondary analyses motivated by the final audit. The primary manuscript conditions are not retuned.

### What remains primary and unchanged

1. The dense encoder is trained from the official SciFact **training qrels only**, for the already fixed 3 epochs, with no validation/test checkpoint selection.
2. The fixed epoch-3 checkpoint is SHA-256 locked before test qrels are loaded and is reloaded with the exact training `TextEncoder` class using `strict=True`.
3. MC-RRF's primary source-count experiment remains depth 100, family size 8, mild 5% candidate-preserving perturbation, RRF `k=60`, output `K=10`.
4. StableRRF's canonical baseline remains `k=60` with BM25 capped at 1000 and dense evidence capped at 100.

### v4 secondary extensions

1. **Deep dense observation.** Regenerate exact dense top-1000 rankings for all four datasets with the same locked checkpoint, while keeping the first 100 ranks as the primary fusion window. StableRRF is then evaluated at dense caps 100/250/500/1000.
2. **Certified-prefix diagnostics.** At each dense cap, report the exact ordered-prefix length even when a complete top-10 certificate is unavailable. This distinguishes "no exact top-10 certificate" from "no exact positions certified".
3. **Task-level adaptive k.** Keep `k=60` as the primary setting. A secondary label-free task calibration uses a deterministic 20% query subset and the already established StableRRF `k` grid `{1,5,10,20,60,100,200}`. It selects one `k` per task from ranking-only certification feasibility, then evaluates that fixed task-level `k` on held-out queries. Relevance qrels are **not used to select k**.
4. **Real-family absolute drift.** Preserve the original signed-effect/Holm analysis and add a clearly labeled post-audit secondary endpoint: absolute nDCG@10 deviation from the clean fusion.

No scientific outcome is used as a crash condition. Protocol/integrity failures still fail loudly; unexpected empirical outcomes are written to the result package.

**Run from top to bottom in a fresh Kaggle GPU session. Do not mix partial outputs from different runs.**


In [ ]:
# Optional dependency check/install for Kaggle.
import importlib, subprocess, sys
required = {
    'datasets': 'datasets',
    'transformers': 'transformers',
    'sklearn': 'scikit-learn',
    'scipy': 'scipy',
    'pandas': 'pandas',
    'sentence_transformers': 'sentence-transformers',
}
missing=[]
for mod,pkg in required.items():
    try: importlib.import_module(mod)
    except Exception: missing.append(pkg)
if missing:
    print('Installing:', missing)
    subprocess.check_call([sys.executable,'-m','pip','install','-q','-U',*missing])
else:
    print('Dependencies already available.')

In [ ]:
from pathlib import Path
from collections import defaultdict
from dataclasses import dataclass
from itertools import permutations, product
import hashlib, json, math, os, random, shutil, subprocess, sys, time, warnings, zipfile

import numpy as np
import pandas as pd
import torch
from scipy.stats import wilcoxon, rankdata

SEED = 41
MODEL_NAME = 'intfloat/e5-base-v2'
EPOCHS = 3
BATCH_SIZE = 4
NEGATIVES_PER_QUERY = 22
MAX_LENGTH = 256
TEMPERATURE = 0.07
HARD_NEG_K = 8
DOC_CHUNK = 32

RRF_K = 60.0
EVAL_K = 10
BM25_TOPK = 1000
DENSE_TOPK = 100  # historical/primary SciFact repair window
DENSE_TOPK_DEEP = 1000  # secondary StableRRF observation extension
STABLE_DENSE_CAP_PRIMARY = 100
STABLE_DENSE_CAPS = [100, 250, 500, 1000]
TASK_K_GRID = [1, 5, 10, 20, 60, 100, 200]
TASK_K_CALIB_FRACTION = 0.20
TASK_K_DENSE_CAP = 100
SPLADE_TOPK = 500
SOURCE_DEPTH = 100
FAMILY_SIZES = [1,2,4,8]
PRIMARY_FAMILY_SIZE = 8
PRIMARY_PERTURB_RATE = 0.05
N_BOOT = 10_000
STAT_SEED = 20260901
ALPHA = 0.05

RUN_SPLADE = True
RUN_SENSITIVITY = True
RUN_STABLERFF = True
RUN_DEEP_DENSE_EXTENSION = True
RUN_TASK_ADAPTIVE_K = True
RUN_REAL_FAMILY_ABS_DRIFT = True

ROOT = Path('/kaggle/working') if Path('/kaggle/working').exists() else Path('.')
OUT = ROOT / 'invariantrrf_scifact_leakage_clean'
TRAIN_ROOT = OUT / 'scifact_train_only'
MODEL_OUT = OUT / 'dense_seed41_fixed_epoch3'
RUN_DIR = OUT / 'runs'
TABLE_DIR = OUT / 'tables'
STAT_DIR = OUT / 'statistics'
for p in [OUT, TRAIN_ROOT/'train', MODEL_OUT, RUN_DIR, TABLE_DIR, STAT_DIR]: p.mkdir(parents=True, exist_ok=True)

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
if torch.cuda.is_available(): torch.cuda.manual_seed_all(SEED)

print('OUT:', OUT)
print('CUDA:', torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 1. Exact historical training helpers

The next three cells reproduce the source notebook's `utils.py`, `bm25_baseline.py`, and `dense_train.py`. The trainer is **not edited**; leakage is prevented by constructing a train-only data root and calling it with `--val_every_epochs 0`.

In [ ]:
%%writefile utils.py
#!/usr/bin/env python3
# utils.py
import os, json, math, time, hashlib, signal, contextlib
from pathlib import Path
from typing import List, Dict, Tuple, Optional, Any, Callable

import torch
from transformers import AutoModel, AutoTokenizer

# ---------------------------
# General utilities
# ---------------------------
def set_seed(seed: int):
    import random, numpy as _np
    random.seed(seed); _np.random.seed(seed); torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)

def read_jsonl(p: Path):
    rows=[]
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            if line.strip():
                rows.append(json.loads(line))
    return rows

def write_json(p: Path, obj: Any):
    p.parent.mkdir(parents=True, exist_ok=True)
    with open(p, "w", encoding="utf-8") as f:
        json.dump(obj, f, indent=2, ensure_ascii=False)

def read_qrels_tsv(p: Path) -> Dict[str, List[str]]:
    q2d={}
    with open(p, "r", encoding="utf-8") as f:
        for line in f:
            sp = line.strip().split("\t")
            if len(sp) >= 2:
                qid, did = sp[0], sp[1]
                q2d.setdefault(qid, []).append(did)
    return q2d

def safe_mkdir(p: Path):
    p.mkdir(parents=True, exist_ok=True)
    return p

# ---------------------------
# Timeouts (POSIX, Kaggle OK)
# ---------------------------
class Timeout(Exception): pass

@contextlib.contextmanager
def time_limit(seconds: int):
    def signal_handler(signum, frame):
        raise Timeout(f"Timed out after {seconds}s")
    if seconds and seconds > 0:
        old = signal.signal(signal.SIGALRM, signal_handler)
        signal.alarm(seconds)
        try:
            yield
        finally:
            signal.alarm(0)
            signal.signal(signal.SIGALRM, old)
    else:
        # no timeout
        yield

# ---------------------------
# Prefix schemes
# ---------------------------
def get_prefixes(prefix_scheme: str = "e5",
                 custom_q: Optional[str] = None,
                 custom_d: Optional[str] = None) -> Tuple[str, str]:
    """
    Supported:
      - e5     -> ("query: ", "passage: ")
      - bge    -> ("", "")  (BGE doesn't require fixed prefixes)
      - none   -> ("", "")
      - custom -> (custom_q or "", custom_d or "")
    """
    scheme = (prefix_scheme or "e5").lower()
    if scheme == "e5":
        return "query: ", "passage: "
    if scheme == "bge":
        return "", ""
    if scheme == "none":
        return "", ""
    if scheme == "custom":
        return (custom_q or ""), (custom_d or "")
    # default to E5
    return "query: ", "passage: "

# ---------------------------
# Robust HF loaders with timeouts
# ---------------------------
def load_tokenizer(model_name: str, timeout_s: int = 120):
    from huggingface_hub import snapshot_download
    with time_limit(timeout_s):
        try:
            tok = AutoTokenizer.from_pretrained(model_name, use_fast=True, trust_remote_code=False)
        except Exception:
            # fallback snapshot
            local = snapshot_download(repo_id=model_name,
                                      allow_patterns=["*tokenizer*", "vocab.*", "*.model", "*.txt", "special_tokens_map.json"])
            tok = AutoTokenizer.from_pretrained(local, use_fast=True, local_files_only=True, trust_remote_code=False)
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token if getattr(tok, "eos_token", None) else "[PAD]"
    return tok

def load_backbone(model_name: str, timeout_s: int = 300):
    from huggingface_hub import snapshot_download
    with time_limit(timeout_s):
        try:
            return AutoModel.from_pretrained(model_name, trust_remote_code=False)
        except Exception:
            local = snapshot_download(repo_id=model_name)
            return AutoModel.from_pretrained(local, local_files_only=True, trust_remote_code=False)

# ---------------------------
# Model / cache fingerprints
# ---------------------------
def file_fingerprint(p: Path) -> str:
    st = p.stat()
    return f"{p.name}:{st.st_size}:{int(st.st_mtime)}"

def tensor_sha(t: torch.Tensor) -> str:
    m = hashlib.sha256()
    m.update(t.detach().cpu().numpy().tobytes())
    return m.hexdigest()[:16]

def model_fingerprint(model_dir: Path) -> str:
    pt = model_dir/"model.pt"
    if pt.exists():
        fp = file_fingerprint(pt)
        return hashlib.sha1(fp.encode("utf-8")).hexdigest()[:16]
    # if only a string name (pretrained)
    return hashlib.sha1(str(model_dir).encode("utf-8")).hexdigest()[:16]

# ---------------------------
# Embedding (with caching)
# ---------------------------
@torch.no_grad()
def embed_texts(enc, tok, texts: List[str], device, max_length=160, batch=64):
    outs=[]
    for i in range(0, len(texts), batch):
        sub=texts[i:i+batch]
        t = tok(sub, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
        t = {k: v.to(device) for k,v in t.items()}
        v = enc(**t)
        outs.append(v.detach().cpu())
    return torch.cat(outs, 0) if outs else torch.empty(0)

@torch.no_grad()
def embed_texts_cached(enc, tok, texts: List[str], device,
                       cache_dir: Path, cache_key: str,
                       max_length=160, batch=64) -> torch.Tensor:
    cache_dir = safe_mkdir(cache_dir)
    cache_file = cache_dir / f"{cache_key}.pt"
    if cache_file.exists():
        try:
            return torch.load(cache_file, map_location="cpu")
        except Exception:
            pass
    embs = embed_texts(enc, tok, texts, device, max_length=max_length, batch=batch)
    torch.save(embs, cache_file)
    return embs

In [ ]:
%%writefile bm25_baseline.py
#!/usr/bin/env python3
# bm25_baseline.py
import math, re
from typing import List, Tuple
def _tok(s: str):
    return re.findall(r"[A-Za-z0-9_]+", (s or "").lower())
class MiniBM25:
    """
    Extremely small BM25 (no external deps). Good enough for paper comparisons
    and ablations when Pyserini isn't available.
    """
    def __init__(self, docs: List[str], k1=1.2, b=0.75):
        self.k1, self.b = k1, b
        self.N = len(docs)
        self.doc_lens=[]
        self.avgdl=0.0
        self.inv={}
        for i,txt in enumerate(docs):
            toks=_tok(txt); self.doc_lens.append(len(toks))
            tf={}
            for t in toks: tf[t]=tf.get(t,0)+1
            for t,c in tf.items():
                self.inv.setdefault(t,{})[i]=c
        self.avgdl=sum(self.doc_lens)/max(1,self.N)
        self.idf={}
        for t,post in self.inv.items():
            df=len(post)
            self.idf[t]=math.log((self.N-df+0.5)/(df+0.5)+1.0)
    def score(self, query: str) -> List[Tuple[int,float]]:
        q=_tok(query); scores=[]
        for i in range(self.N):
            toks_len = self.doc_lens[i]
            s=0.0
            for t in q:
                c = self.inv.get(t, {}).get(i, 0)
                if c<=0: continue
                idf=self.idf.get(t,0.0)
                denom=c + self.k1*(1 - self.b + self.b*toks_len/self.avgdl)
                s += idf * (c*(self.k1+1))/max(1e-6,denom)
            if s!=0.0: scores.append((i, s))
        scores.sort(key=lambda x: x[1], reverse=True)
        return scores

In [ ]:
%%writefile dense_train.py
#!/usr/bin/env python3
# dense_train.py
import os, time, json, argparse, shutil, random
from pathlib import Path
from typing import Dict, List, Optional

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from transformers import get_linear_schedule_with_warmup

from utils import (
    set_seed, read_jsonl, read_qrels_tsv,
    load_backbone, load_tokenizer, get_prefixes
)
from bm25_baseline import MiniBM25

# ---------------- pooling & encoder ----------------
class GeMPooler(nn.Module):
    def __init__(self, p=3.0, eps=1e-6): super().__init__(); self.p=nn.Parameter(torch.tensor(float(p))); self.eps=eps
    def forward(self, x, mask):
        mask = mask.unsqueeze(-1).type_as(x)
        x = x.clamp(min=self.eps); x=(x**self.p)*mask
        denom = mask.sum(dim=1).clamp_min(self.eps); x = x.sum(dim=1)/denom
        return x.clamp(min=self.eps)**(1.0/self.p)

class TextEncoder(nn.Module):
    def __init__(self, model_name:str, use_gem=True, proj=False):
        super().__init__()
        self.backbone = load_backbone(model_name)
        self.use_gem = use_gem
        self.pool = GeMPooler() if use_gem else None
        self.proj = nn.Linear(self.backbone.config.hidden_size, self.backbone.config.hidden_size, bias=False) if proj else None
    def forward(self, input_ids=None, attention_mask=None, **kwargs):
        safe={"input_ids": input_ids, "attention_mask": attention_mask}
        out = self.backbone(**safe, return_dict=True).last_hidden_state
        if self.use_gem:
            pooled = self.pool(out, attention_mask)
        else:
            mask = attention_mask.unsqueeze(-1).type_as(out)
            pooled = (out*mask).sum(1)/mask.sum(1).clamp_min(1e-6)
        if self.proj is not None:
            pooled = self.proj(pooled)
        return F.normalize(pooled, dim=-1)

# ---------------- dataset (with BM25 hard-neg caching) ----------------
def _sig(path: Path) -> str:
    try:
        st = path.stat(); return f"{st.st_size}-{int(st.st_mtime)}"
    except Exception:
        return "na"

class MPDRDataset(Dataset):
    def __init__(self, root:str, fractions:float=1.0, max_pos_per_query:int=0,
                 noise_rate:float=0.0, all_docs=None,
                 hard_negatives:str="none", hard_neg_k:int=4,
                 hard_cache:bool=True, hard_cache_dir:Optional[str]=None, hard_cache_rebuild:bool=False):
        root=Path(root)
        self.qs = read_jsonl(root/"train"/"queries.jsonl")
        self.ds = read_jsonl(root/"train"/"docs.jsonl")
        self.qrels = read_qrels_tsv(root/"train"/"qrels.tsv")
        if fractions < 1.0:
            keep = max(1, int(round(len(self.qs)*fractions)))
            self.qs = self.qs[:keep]
        self.doc_map = {d["id"]: d["text"] for d in self.ds}
        self.doc_ids = list(self.doc_map.keys())
        self.max_pos = max_pos_per_query
        self.noise_rate = noise_rate
        self.all_texts = all_docs if all_docs is not None else [d["text"] for d in self.ds]

        self.hard_negatives = hard_negatives
        self.hard_neg_k = hard_neg_k
        self.hard_cache: Dict[str, List[str]] = {}

        if self.hard_negatives == "bm25":
            cache_dir = Path(hard_cache_dir) if hard_cache_dir else (root/"cache"/"hardneg_bm25")
            cache_dir.mkdir(parents=True, exist_ok=True)
            meta = {
                "docs_sig": _sig(root/"train"/"docs.jsonl"),
                "queries_sig": _sig(root/"train"/"queries.jsonl"),
                "hard_neg_k": self.hard_neg_k,
            }
            cache_file = cache_dir/f"hardnegs_k{self.hard_neg_k}.json"
            loaded=False
            if hard_cache and cache_file.exists() and not hard_cache_rebuild:
                try:
                    blob = json.loads(cache_file.read_text("utf-8"))
                    if blob.get("meta")==meta and isinstance(blob.get("data"), dict):
                        self.hard_cache = {str(k): [str(x) for x in v] for k,v in blob["data"].items()}
                        if len(self.hard_cache)>=int(0.9*len(self.qs)):
                            print(f"[hard-neg] loaded cached hard negatives from {cache_file}")
                            loaded=True
                        else:
                            print("[hard-neg] cache coverage too low; recomputing")
                except Exception as e:
                    print("[hard-neg] cache load failed -> recompute:", e)

            if not loaded:
                print("[hard-neg] precomputing BM25 negatives (first time or rebuild)")
                bm25 = MiniBM25(self.all_texts)
                for ex in self.qs:
                    qid, qtext = ex["id"], ex["text"]
                    scored = bm25.score(qtext)
                    pos_set = set(self.qrels.get(qid, []))
                    hard_ids=[]
                    for j,_ in scored:
                        did = self.doc_ids[j] if j < len(self.doc_ids) else None
                        if did is None or did in pos_set: continue
                        hard_ids.append(did)
                        if len(hard_ids) >= max(50, self.hard_neg_k*5): break
                    self.hard_cache[qid]=hard_ids
                if hard_cache:
                    try:
                        cache_file.write_text(json.dumps({"meta": meta, "data": self.hard_cache}), encoding="utf-8")
                        print(f"[hard-neg] cache written to {cache_file}")
                    except Exception as e:
                        print("[hard-neg] failed to write cache:", e)

    def __len__(self): return len(self.qs)

    def __getitem__(self, idx):
        q = self.qs[idx]
        qid = q["id"]; qtext = q["text"]
        pos_ids = list(self.qrels.get(qid, []))
        if self.max_pos>0 and len(pos_ids)>self.max_pos:
            pos_ids = pos_ids[:self.max_pos]
        pos_texts = [self.doc_map[pid] for pid in pos_ids if pid in self.doc_map]
        if self.noise_rate>0 and random.random() < self.noise_rate and len(pos_texts)>0:
            repl = random.choice(self.all_texts)
            pos_texts[0] = repl
        hard = []
        if self.hard_negatives == "bm25":
            hard_ids = (self.hard_cache.get(qid, []) or [])[:self.hard_neg_k]
            for did in hard_ids:
                txt = self.doc_map.get(did)
                if txt is not None:
                    hard.append(txt)
        return {"qid": qid, "qtext": qtext, "pos_texts": pos_texts, "hard_negs": hard}

# ---------------- collate + loss ----------------
def collate(batch, tokenizer, max_length, negatives_per_query, all_doc_texts, mode, q_prefix, d_prefix):
    q_texts=[q_prefix + b["qtext"] for b in batch]
    pos_texts=[]; pos_slices=[]
    rng = random.Random(777 + len(batch))
    neg_texts=[]
    for b in batch:
        pts=[d_prefix + t for t in b["pos_texts"]]
        start=len(pos_texts); pos_texts.extend(pts); end=len(pos_texts); pos_slices.append((start,end))
        hards = [d_prefix + t for t in b.get("hard_negs", [])]
        need = max(0, negatives_per_query - len(hards))
        rnds = [d_prefix + rng.choice(all_doc_texts) for _ in range(need)]
        neg_texts.extend(hards + rnds)
    if mode=="single":
        new_pos=[]; new_slices=[]; cursor=0
        for (s,e) in pos_slices:
            if e-s>0:
                new_pos.append(pos_texts[s]); new_slices.append((cursor,cursor+1)); cursor+=1
            else:
                new_slices.append((cursor,cursor))
        pos_texts = new_pos; pos_slices = new_slices
    q_tok = tokenizer(q_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
    d_tok = tokenizer(pos_texts + neg_texts, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
    return {
        "q_tok": q_tok, "d_tok": d_tok, "pos_slices": pos_slices,
        "num_pos_total": len(pos_texts), "num_neg_total": len(neg_texts)
    }

def supcon_multi_positive(q, P, neg, tau, w=None):
    if P.size(0)==0: return q.new_tensor(0.0)
    pos_logits = (q @ P.t())/tau
    if neg.numel()>0:
        denom = torch.cat([pos_logits, (q @ neg.t())/tau], 0)
    else:
        denom = pos_logits
    log_den = torch.logsumexp(denom, 0)
    if w is None:
        return - (pos_logits - log_den).mean()
    w = w / w.sum().clamp_min(1e-6)
    return - (w * (pos_logits - log_den)).sum()

# ---------------- dev eval (nDCG@10) ----------------
def _ndcg_at_k(flags, k):
    flags=flags[:k]; dcg=0.0
    import math as _m
    for i,f in enumerate(flags,1):
        if f: dcg += 1.0/_m.log2(i+1)
    idcg=0.0
    for i in range(1, min(k, sum(flags))+1):
        idcg += 1.0/_m.log2(i+1)
    return (dcg/idcg) if idcg>0 else 0.0

@torch.no_grad()
def eval_dev_ndcg10(enc, tok, data_root, max_length=160, q_prefix="", d_prefix="", batch=64, device=None):
    dev_dir = Path(data_root)/"dev"
    qrels_p = dev_dir/"qrels.tsv"
    if not qrels_p.exists(): return None
    queries = read_jsonl(dev_dir/"queries.jsonl")
    docs    = read_jsonl(dev_dir/"docs.jsonl")
    qrels   = read_qrels_tsv(qrels_p)
    device = device or (torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu"))
    enc.eval()

    def embed(texts:List[str]):
        outs=[]
        for i in range(0, len(texts), batch):
            sub=texts[i:i+batch]
            t = tok(sub, padding=True, truncation=True, max_length=max_length, return_tensors="pt")
            t = {k: v.to(device) for k,v in t.items()}
            v = enc(**t)
            outs.append(v.detach().cpu())
        return torch.cat(outs, 0) if outs else torch.empty(0)

    q_texts = [(q_prefix or "") + q["text"] for q in queries]
    d_texts = [(d_prefix or "") + d["text"] for d in docs]
    q_emb = embed(q_texts); d_emb = embed(d_texts)
    scores = (q_emb @ d_emb.t()).cpu()
    ndcgs = []
    for i,q in enumerate(queries):
        ranked = torch.topk(scores[i], k=min(100, scores.size(1))).indices.tolist()
        ranked_ids = [docs[j]["id"] for j in ranked]
        gold = set(qrels.get(q["id"], []))
        flags = [1 if did in gold else 0 for did in ranked_ids]
        ndcgs.append(_ndcg_at_k(flags, 10))
    return float(sum(ndcgs)/max(1,len(ndcgs)))

# ---------------- training ----------------
def train(
    data_root:str, out_dir:str, model_name:str="intfloat/e5-small-v2",
    mode:str="multi",
    epochs:int=3, batch_size:int=8, negatives_per_query:int=20, max_length:int=160,
    temperature:float=0.07, supcon_weighting:str="none",
    prefix_scheme:str="e5", query_prefix:str="", doc_prefix:str="",
    seed:int=41, lr:float=3e-5, grad_accum_steps:int=1,
    doc_chunk:int=0, amp:bool=False, grad_checkpointing:bool=False,
    max_pos_per_query:int=0, fractions:float=1.0, noise_rate:float=0.0,
    hard_negatives:str="none", hard_neg_k:int=4,
    val_every_epochs:int=1,
    hard_cache:bool=True, hard_cache_dir:Optional[str]=None, hard_cache_rebuild:bool=False
):
    assert mode in ("single","multi")
    if negatives_per_query < 0: raise ValueError("negatives_per_query must be >= 0")
    if max_length <= 0 or max_length > 4096: raise ValueError("max_length out of range")
    if supcon_weighting not in ("none","cosine","bm25"): raise ValueError("invalid supcon_weighting")

    set_seed(seed)
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    os.environ.setdefault("PYTORCH_CUDA_ALLOC_CONF", "expandable_segments:True,max_split_size_mb:64")
    print(f"[Device] {device}")

    out = Path(out_dir); out.mkdir(parents=True, exist_ok=True)

    # prefixes
    if not (query_prefix or doc_prefix):
        query_prefix, doc_prefix = get_prefixes(prefix_scheme, query_prefix, doc_prefix)

    # Data
    all_docs = [d["text"] for d in read_jsonl(Path(data_root)/"train"/"docs.jsonl")]
    ds = MPDRDataset(
        data_root, fractions=fractions, max_pos_per_query=max_pos_per_query,
        noise_rate=noise_rate, all_docs=all_docs,
        hard_negatives=hard_negatives, hard_neg_k=hard_neg_k,
        hard_cache=hard_cache, hard_cache_dir=hard_cache_dir, hard_cache_rebuild=hard_cache_rebuild
    )
    tok = load_tokenizer(model_name)

    def _collate(batch):
        return collate(batch, tok, max_length, negatives_per_query, all_docs, mode, query_prefix, doc_prefix)

    loader = DataLoader(ds, batch_size=batch_size, shuffle=True, drop_last=False, collate_fn=_collate)

    # Model
    enc = TextEncoder(model_name, use_gem=True, proj=False).to(device)
    if grad_checkpointing:
        try:
            enc.backbone.gradient_checkpointing_enable()
            if hasattr(enc.backbone.config, "use_cache"): enc.backbone.config.use_cache=False
            print("[grad_ckpt] enabled")
        except Exception as e:
            print("[grad_ckpt] failed:", e)

    opt = torch.optim.AdamW(enc.parameters(), lr=lr)
    total_steps = max(1, (len(loader)*epochs)//max(1,grad_accum_steps))
    sched = get_linear_schedule_with_warmup(opt, num_warmup_steps=max(10,total_steps//20), num_training_steps=total_steps)
    scaler = torch.amp.GradScaler("cuda", enabled=amp)

    best_score = -1.0
    best_dir = out/"best"; best_dir.mkdir(parents=True, exist_ok=True)

    for ep in range(1, epochs+1):
        for it, batch in enumerate(loader, 1):
            enc.train()
            q_tok = {k: v.to(device) for k,v in batch["q_tok"].items()}
            d_tok = {k: v.to(device) for k,v in batch["d_tok"].items()}
            pos_slices = batch["pos_slices"]
            num_pos_total = batch["num_pos_total"]; num_neg_total = batch["num_neg_total"]

            t0 = time.perf_counter()
            with torch.amp.autocast("cuda", enabled=amp, dtype=torch.float16):
                q_emb = enc(**q_tok)
                def encode_chunked(tok:Dict[str,torch.Tensor], chunk:int=0):
                    if chunk<=0: return enc(**{k: v.to(device) for k,v in tok.items()})
                    outs=[]; ids=tok["input_ids"]; msk=tok["attention_mask"]
                    for s in range(0, ids.size(0), chunk):
                        sub = {"input_ids": ids[s:s+chunk].to(device), "attention_mask": msk[s:s+chunk].to(device)}
                        outs.append(enc(**sub))
                    return torch.cat(outs, 0)
                d_emb = encode_chunked(d_tok, doc_chunk)
                pos_emb = d_emb[:num_pos_total]
                neg_emb = d_emb[num_pos_total:] if num_neg_total>0 else torch.empty((0, d_emb.size(1)), device=device)

                losses=[]
                for i, (s,e) in enumerate(pos_slices):
                    q = q_emb[i]; P = pos_emb[s:e]
                    if P.size(0)==0: losses.append(q.new_tensor(0.0)); continue
                    w=None
                    if mode=="multi":
                        if supcon_weighting=="cosine":
                            with torch.no_grad(): w = (q @ P.t()).clamp_min(0)
                        elif supcon_weighting=="bm25":
                            with torch.no_grad(): w = (q @ P.t()).clamp_min(0)
                    L = supcon_multi_positive(q, P if mode=="multi" else P[:1], neg_emb, temperature, w)
                    losses.append(L)
                loss = torch.stack(losses).mean()

            scaler.scale(loss / max(1,grad_accum_steps)).backward()
            if (it % grad_accum_steps)==0:
                scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); sched.step()

            if it % 50 == 0:
                mem = torch.cuda.max_memory_allocated()/1e9 if torch.cuda.is_available() else 0.0
                toks = q_tok["input_ids"].numel() + d_tok["input_ids"].numel()
                t1 = time.perf_counter()
                print(f"[epoch {ep} step {it}] loss={loss.item():.4f} time/step={(t1-t0):.3f}s tok/s={toks/max(1e-9,(t1-t0)):.0f} peakVRAM={mem:.2f}GB")

        if val_every_epochs>0 and (ep % val_every_epochs)==0:
            score = eval_dev_ndcg10(enc, tok, data_root, max_length=max_length, q_prefix=query_prefix, d_prefix=doc_prefix, device=device)
            if score is None:
                print("[eval] No dev qrels found; skipping best-model selection.")
            else:
                print(f"[eval] epoch {ep} dev nDCG@10 = {score:.4f}")
                if score > best_score:
                    best_score = score
                    for f in best_dir.glob("*"):
                        if f.is_file(): f.unlink()
                    torch.save({"state_dict": enc.state_dict(), "model_name": model_name, "use_gem": True, "proj": False}, (best_dir/"model.pt"))
                    tok.save_pretrained(best_dir.as_posix())
                    with open(best_dir/"metric.txt","w") as wf:
                        wf.write(f"nDCG@10\t{score:.6f}\n")
                    print(f"[best] updated -> {best_dir} (nDCG@10={score:.4f})")

    final = out/"final"; final.mkdir(parents=True, exist_ok=True)
    if best_score >= 0.0 and (best_dir/"model.pt").exists():
        shutil.copy2(best_dir/"model.pt", final/"model.pt")
        for p in best_dir.iterdir():
            if p.is_file() and p.name!="model.pt":
                shutil.copy2(p, final/p.name)
        print(f"[done] best model copied to {final} (nDCG@10={best_score:.4f})")
    else:
        torch.save({"state_dict": enc.state_dict(), "model_name": model_name, "use_gem": True, "proj": False}, final/"model.pt")
        tok.save_pretrained(final.as_posix())
        print(f"[done] final model saved to {final} (no dev evaluation available)")

def main():
    ap = argparse.ArgumentParser()
    ap.add_argument("--data_root", required=True)
    ap.add_argument("--out_dir", required=True)
    ap.add_argument("--model_name", default="intfloat/e5-small-v2")
    ap.add_argument("--mode", default="multi", choices=["single","multi"])
    ap.add_argument("--epochs", type=int, default=3)
    ap.add_argument("--batch_size", type=int, default=8)
    ap.add_argument("--negatives_per_query", type=int, default=20)
    ap.add_argument("--max_length", type=int, default=160)
    ap.add_argument("--temperature", type=float, default=0.07)
    ap.add_argument("--supcon_weighting", default="none", choices=["none","cosine","bm25"])
    ap.add_argument("--prefix_scheme", default="e5", choices=["e5","bge","none","custom"])
    ap.add_argument("--query_prefix", type=str, default="")
    ap.add_argument("--doc_prefix", type=str, default="")
    ap.add_argument("--seed", type=int, default=41)
    ap.add_argument("--lr", type=float, default=3e-5)
    ap.add_argument("--grad_accum_steps", type=int, default=1)
    ap.add_argument("--doc_chunk", type=int, default=0)
    ap.add_argument("--amp", action="store_true")
    ap.add_argument("--grad_checkpointing", action="store_true")
    ap.add_argument("--max_pos_per_query", type=int, default=0)
    ap.add_argument("--fractions", type=float, default=1.0)
    ap.add_argument("--noise_rate", type=float, default=0.0)
    ap.add_argument("--hard_negatives", default="none", choices=["none","bm25"])
    ap.add_argument("--hard_neg_k", type=int, default=4)
    ap.add_argument("--val_every_epochs", type=int, default=1)
    ap.add_argument("--hard_cache", action="store_true", default=True)
    ap.add_argument("--no_hard_cache", dest="hard_cache", action="store_false")
    ap.add_argument("--hard_cache_dir", type=str, default=None)
    ap.add_argument("--hard_cache_rebuild", action="store_true")
    args = ap.parse_args()

    train(
        data_root=args.data_root, out_dir=args.out_dir, model_name=args.model_name,
        mode=args.mode, epochs=args.epochs, batch_size=args.batch_size,
        negatives_per_query=args.negatives_per_query, max_length=args.max_length,
        temperature=args.temperature, supcon_weighting=args.supcon_weighting,
        prefix_scheme=args.prefix_scheme, query_prefix=args.query_prefix, doc_prefix=args.doc_prefix,
        seed=args.seed, lr=args.lr, grad_accum_steps=args.grad_accum_steps, doc_chunk=args.doc_chunk,
        amp=args.amp, grad_checkpointing=args.grad_checkpointing,
        max_pos_per_query=args.max_pos_per_query, fractions=args.fractions, noise_rate=args.noise_rate,
        hard_negatives=args.hard_negatives, hard_neg_k=args.hard_neg_k,
        val_every_epochs=args.val_every_epochs,
        hard_cache=args.hard_cache, hard_cache_dir=args.hard_cache_dir, hard_cache_rebuild=args.hard_cache_rebuild
    )

if __name__=="__main__":
    main()

## 2. Build a SciFact **train-only** training root

The official test qrels are deliberately not loaded in this section.

In [ ]:
from datasets import load_dataset

def norm_id(x):
    if x is None: return None
    if isinstance(x,(int,float)): return str(int(x))
    return str(x)

def qrel_pairs(ds_split):
    pairs=[]
    for ex in ds_split:
        qid = norm_id(ex.get('query-id') or ex.get('query_id') or ex.get('_id') or ex.get('qid'))
        if 'qrels' in ex and isinstance(ex['qrels'],dict):
            for did,score in ex['qrels'].items():
                rel=int(score) if score else 0
                if rel>0: pairs.append((qid,norm_id(did),rel))
            continue
        did = ex.get('corpus-id') or ex.get('corpus_id') or ex.get('doc-id') or ex.get('doc_id') or ex.get('pid')
        rel_raw = ex.get('score') or ex.get('relevance') or ex.get('rel') or 1
        rel=int(rel_raw) if rel_raw else 1
        if rel>0: pairs.append((qid,norm_id(did),rel))
    return pairs

def write_jsonl(path, rows):
    path.parent.mkdir(parents=True,exist_ok=True)
    with open(path,'w',encoding='utf-8') as f:
        for rid,text in rows:
            f.write(json.dumps({'id':str(rid),'text':text or ''},ensure_ascii=False)+'\n')

def write_qrels(path,pairs):
    path.parent.mkdir(parents=True,exist_ok=True)
    seen=set()
    with open(path,'w',encoding='utf-8') as f:
        for qid,did,rel in pairs:
            if qid is None or did is None or (qid,did) in seen: continue
            seen.add((qid,did)); f.write(f'{qid}\t{did}\t{int(rel)}\n')

# Qrels: load TRAIN ONLY here. No test qrels are touched before the model lock.
train_qrels_ds = load_dataset('BeIR/scifact-qrels', split='train')
corpus_ds = load_dataset('BeIR/scifact', 'corpus')['corpus']
queries_ds = load_dataset('BeIR/scifact', 'queries')['queries']

# Corpus/query text is label-free. Only official train qids become training examples.
docs=[]; doc_ids=set()
for ex in corpus_ds:
    did=norm_id(ex.get('_id') or ex.get('doc-id') or ex.get('corpus-id'))
    title=(ex.get('title') or '').strip(); text=(ex.get('text') or '').strip()
    full=(title + ('\n\n' if title and text else '') + text).strip()
    docs.append((did,full)); doc_ids.add(did)
qmap={}
for ex in queries_ds:
    qid=norm_id(ex.get('_id') or ex.get('query-id') or ex.get('query_id') or ex.get('qid'))
    if qid is not None: qmap[qid]=(ex.get('text') or '').strip()

train_pairs=[x for x in qrel_pairs(train_qrels_ds) if x[0] in qmap and x[1] in doc_ids]
train_qids=[]; seen=set()
for qid,_,_ in train_pairs:
    if qid not in seen: seen.add(qid); train_qids.append(qid)

# Clean any stale local training root and write only train/. No dev/ directory is created.
if TRAIN_ROOT.exists(): shutil.rmtree(TRAIN_ROOT)
(TRAIN_ROOT/'train').mkdir(parents=True,exist_ok=True)
write_jsonl(TRAIN_ROOT/'train'/'docs.jsonl',docs)
write_jsonl(TRAIN_ROOT/'train'/'queries.jsonl',[(q,qmap[q]) for q in train_qids])
write_qrels(TRAIN_ROOT/'train'/'qrels.tsv',train_pairs)

assert not (TRAIN_ROOT/'dev'/'qrels.tsv').exists(), 'Leakage guard failed: dev/test qrels exist before training.'
print({'documents':len(docs),'train_queries':len(train_qids),'train_qrel_pairs':len(train_pairs),'dev_qrels_present':False})

## 3. Train fixed epoch-3 checkpoint with **no metric-based selection**

In [ ]:
# Preserve the historical seed-41 recipe, adding only --val_every_epochs 0.
if MODEL_OUT.exists(): shutil.rmtree(MODEL_OUT)
MODEL_OUT.mkdir(parents=True,exist_ok=True)
cmd=[
    sys.executable,'-u','dense_train.py',
    '--data_root',str(TRAIN_ROOT),
    '--out_dir',str(MODEL_OUT),
    '--model_name',MODEL_NAME,
    '--mode','multi',
    '--epochs',str(EPOCHS),
    '--batch_size',str(BATCH_SIZE),
    '--negatives_per_query',str(NEGATIVES_PER_QUERY),
    '--max_length',str(MAX_LENGTH),
    '--temperature',str(TEMPERATURE),
    '--prefix_scheme','e5',
    '--seed',str(SEED),
    '--hard_negatives','bm25',
    '--hard_neg_k',str(HARD_NEG_K),
    '--amp',
    '--doc_chunk',str(DOC_CHUNK),
    '--val_every_epochs','0',
]
print(' '.join(cmd))
subprocess.check_call(cmd)

FINAL_MODEL = MODEL_OUT/'final'
assert (FINAL_MODEL/'model.pt').exists()
assert not (TRAIN_ROOT/'dev'/'qrels.tsv').exists()
print('Fixed epoch-3 checkpoint saved:', FINAL_MODEL)

## 4. Lock the checkpoint **before** loading test qrels

In [ ]:
def sha256_file(path, chunk=1024*1024):
    h=hashlib.sha256()
    with open(path,'rb') as f:
        while True:
            b=f.read(chunk)
            if not b: break
            h.update(b)
    return h.hexdigest()

model_hash=sha256_file(FINAL_MODEL/'model.pt')
lock={
    'dataset':'SciFact',
    'model_name':MODEL_NAME,
    'seed':SEED,
    'selection_policy':'fixed final epoch; no validation/test metric selection',
    'epochs_fixed':EPOCHS,
    'val_every_epochs':0,
    'model_sha256':model_hash,
    'training_qrels':'official BEIR SciFact train split only',
    'test_qrels_loaded_before_lock':False,
}
LOCK_PATH=OUT/'MODEL_LOCK.json'
LOCK_PATH.write_text(json.dumps(lock,indent=2),encoding='utf-8')
print(json.dumps(lock,indent=2))

## 5. Only now load the official SciFact test qrels

In [ ]:
assert LOCK_PATH.exists() and json.loads(LOCK_PATH.read_text())['model_sha256']==sha256_file(FINAL_MODEL/'model.pt')

test_qrels_ds = load_dataset('BeIR/scifact-qrels', split='test')
test_pairs=[x for x in qrel_pairs(test_qrels_ds) if x[0] in qmap and x[1] in doc_ids]
test_qids=[]; seen=set()
for qid,_,_ in test_pairs:
    if qid not in seen: seen.add(qid); test_qids.append(qid)
assert set(train_qids).isdisjoint(test_qids), 'Train/test qid overlap.'
qrels=defaultdict(dict)
for qid,did,rel in test_pairs: qrels[str(qid)][str(did)]=float(rel)
qrels=dict(qrels)
TEST_QMAP={q:qmap[q] for q in test_qids}
DMAP=dict(docs)
print({'test_queries':len(test_qids),'test_qrel_pairs':len(test_pairs),'train_test_qids_disjoint':True})
assert len(test_qids)==300, f'Expected 300 SciFact test queries, got {len(test_qids)}'

## 6. Generate the leakage-clean dense top-100 run with **strict checkpoint loading**

This cell intentionally imports the same `TextEncoder` class used during training. The saved state dict therefore uses the same `backbone.*` key namespace at save and load time. `strict=True` is mandatory: any missing or unexpected parameter aborts the notebook before retrieval.


In [ ]:
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoTokenizer
from utils import get_prefixes
from dense_train import TextEncoder


def load_fixed_model(model_dir):
    """Reload the saved checkpoint with the exact training architecture.

    Fail loudly if *any* checkpoint key is not consumed. This prevents the
    historical `backbone.*` -> `bb.*` namespace mismatch from silently falling
    back to freshly downloaded E5 backbone weights.
    """
    model_dir = Path(model_dir)
    blob = torch.load(model_dir / 'model.pt', map_location='cpu')

    enc = TextEncoder(
        blob['model_name'],
        use_gem=bool(blob.get('use_gem', True)),
        proj=bool(blob.get('proj', False)),
    )

    load_result = enc.load_state_dict(blob['state_dict'], strict=True)
    assert len(load_result.missing_keys) == 0, load_result.missing_keys
    assert len(load_result.unexpected_keys) == 0, load_result.unexpected_keys

    # Independent namespace/content audit: a substantial saved transformer
    # tensor must exist under `backbone.*` and must match the loaded model
    # byte-for-byte before any ranking is generated.
    loaded_sd = enc.state_dict()
    backbone_keys = [
        k for k, v in blob['state_dict'].items()
        if k.startswith('backbone.') and k in loaded_sd and torch.is_tensor(v)
    ]
    assert backbone_keys, 'No saved backbone.* tensors found in checkpoint.'
    probe_key = next((k for k in backbone_keys if blob['state_dict'][k].numel() > 1000), backbone_keys[0])
    assert torch.equal(
        loaded_sd[probe_key].detach().cpu(),
        blob['state_dict'][probe_key].detach().cpu(),
    ), f'Checkpoint tensor mismatch after strict load: {probe_key}'

    tok = AutoTokenizer.from_pretrained(str(model_dir))
    if tok.pad_token is None:
        tok.pad_token = tok.eos_token if getattr(tok, 'eos_token', None) else '[PAD]'

    print('STRICT CHECKPOINT LOAD: PASS')
    print('  missing keys   :', list(load_result.missing_keys))
    print('  unexpected keys:', list(load_result.unexpected_keys))
    print('  verified tensor:', probe_key, tuple(loaded_sd[probe_key].shape))
    return enc, tok


@torch.no_grad()
def embed_batches(enc, tok, texts, device, batch=64, max_length=256):
    outs = []
    for i in range(0, len(texts), batch):
        t = tok(
            texts[i:i+batch],
            padding=True,
            truncation=True,
            max_length=max_length,
            return_tensors='pt',
        )
        t = {k: v.to(device) for k, v in t.items()}
        outs.append(enc(**t).detach().cpu())
    return torch.cat(outs, 0)


def rank_dense_fixed(qmap, dmap, model_dir, top_k=100):
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
    enc, tok = load_fixed_model(model_dir)
    enc.to(device).eval()

    qpref, dpref = get_prefixes('e5')
    qids = list(qmap)
    docids = list(dmap)

    Q = embed_batches(
        enc, tok, [qpref + qmap[q] for q in qids],
        device, max_length=MAX_LENGTH,
    )
    D = embed_batches(
        enc, tok, [dpref + dmap[d] for d in docids],
        device, max_length=MAX_LENGTH,
    )

    scores = (Q @ D.t()).cpu()
    run = {}
    for i, q in enumerate(qids):
        vals, idx = torch.topk(scores[i], k=min(top_k, scores.size(1)))
        run[q] = [
            (str(docids[j]), float(vals[k]))
            for k, j in enumerate(idx.tolist())
        ]

    del Q, D, scores, enc
    if torch.cuda.is_available():
        torch.cuda.empty_cache()
    return run


dense_run = rank_dense_fixed(TEST_QMAP, DMAP, FINAL_MODEL, top_k=DENSE_TOPK)
(RUN_DIR / 'run_dense_clean_seed41_epoch3_STRICT.json').write_text(
    json.dumps(dense_run), encoding='utf-8'
)
print('Dense run:', len(dense_run), 'queries; max depth', max(map(len, dense_run.values())))


## 7. Evaluation helpers and exact InvariantRRF v5 BM25 family

In [ ]:
def dcg_at_k(docids, rels, k=10):
    s=0.0
    for rank,d in enumerate(docids[:k],1):
        rel=float(rels.get(str(d),0.0)); s+=(2.0**rel-1.0)/math.log2(rank+1.0)
    return s

def ndcg_at_k(docids, rels, k=10):
    dcg=dcg_at_k(docids,rels,k); ideal=sorted((float(v) for v in rels.values()),reverse=True)[:k]
    if not ideal: return 0.0
    idcg=sum((2.0**rel-1.0)/math.log2(i+2.0) for i,rel in enumerate(ideal))
    return dcg/idcg if idcg>0 else 0.0

def prefix_docs(seq,L):
    docs=[str(x[0]) if isinstance(x,(tuple,list)) else str(x) for x in seq]
    L=max(0,min(int(L),len(docs))); pref=docs[:L]
    if len(pref)!=len(set(pref)): raise ValueError('Ranking prefix contains duplicate IDs')
    return pref

def rbo_finite(a,b,p=0.9,depth=None):
    max_available=max(len(a),len(b)); depth=max_available if depth is None else min(int(depth),max_available)
    if depth<=0:return 1.0
    A=set();B=set();num=0.;den=0.
    for d in range(1,depth+1):
        if d<=len(a):A.add(a[d-1])
        if d<=len(b):B.add(b[d-1])
        w=p**(d-1); num+=(len(A&B)/d)*w; den+=w
    return num/den if den else 1.0

BM25_CONFIGS={'bm25':(1.5,0.75),'bm25_lowb':(1.2,0.35),'bm25_highb':(1.8,0.90)}

def generate_bm25_family(qmap,dmap,top_k=1000,configs=None):
    from sklearn.feature_extraction.text import CountVectorizer
    configs=configs or BM25_CONFIGS; docids=list(dmap); docs=[dmap[d] for d in docids]
    vectorizer=CountVectorizer(lowercase=True,token_pattern=r'(?u)\b[A-Za-z0-9_]+\b')
    X=vectorizer.fit_transform(docs).tocsr(); Xc=X.tocsc(); N=X.shape[0]
    dl=np.asarray(X.sum(axis=1)).ravel().astype(np.float64); avgdl=float(dl.mean()) if len(dl) else 1.0
    df=np.diff(Xc.indptr).astype(np.float64); idf=np.log1p((N-df+0.5)/(df+0.5))
    qids=list(qmap); qX=vectorizer.transform([qmap[q] for q in qids]).tocsr(); runs={name:{} for name in configs}; K=min(int(top_k),N)
    for qi,qid in enumerate(qids):
        terms=np.unique(qX.indices[qX.indptr[qi]:qX.indptr[qi+1]])
        for name,(k1,b) in configs.items():
            scores=np.zeros(N,dtype=np.float64); norm=k1*(1.0-b+b*dl/max(avgdl,1e-12))
            for t in terms:
                st,en=Xc.indptr[t],Xc.indptr[t+1]; rows=Xc.indices[st:en]; tf=Xc.data[st:en].astype(np.float64)
                scores[rows]+=idf[t]*(tf*(k1+1.0)/(tf+norm[rows]))
            positive=np.flatnonzero(scores>0)
            if len(positive)>=K:
                ps=scores[positive]; take=np.argpartition(-ps,K-1)[:K]; idx=positive[take].tolist()
            else:
                idx=positive.tolist(); need=K-len(idx)
                if need>0:
                    posset=set(idx); zeros=[j for j in range(N) if j not in posset]; zeros.sort(key=lambda j:str(docids[j])); idx.extend(zeros[:need])
            idx=sorted(idx,key=lambda j:(-scores[j],str(docids[j])))
            runs[name][str(qid)]=[(str(docids[j]),float(scores[j])) for j in idx]
    return runs

bm25_runs=generate_bm25_family(TEST_QMAP,DMAP,top_k=BM25_TOPK)
for name,run in bm25_runs.items(): (RUN_DIR/f'run_{name}_top1000.json').write_text(json.dumps(run),encoding='utf-8')

baseline_dense=np.mean([ndcg_at_k([d for d,_ in dense_run[q]],qrels[q],10) for q in test_qids])
baseline_bm25=np.mean([ndcg_at_k([d for d,_ in bm25_runs['bm25'][q]],qrels[q],10) for q in test_qids])
print({'dense_nDCG@10':baseline_dense,'bm25_nDCG@10':baseline_bm25})

## 8. Generate/reuse EnsembleDistil SPLADE for the controlled BM25-family and 3-source StableRRF analyses

In [ ]:
splade_run=None
if RUN_SPLADE:
    from sentence_transformers import SparseEncoder
    from sentence_transformers.util import semantic_search, dot_score
    SPLADE_MODEL='naver/splade-cocondenser-ensembledistil'
    splade_path=RUN_DIR/'run_splade_ensemble_top500.json'
    if splade_path.exists():
        splade_run=json.loads(splade_path.read_text())
        print('Reused',splade_path)
    else:
        model=SparseEncoder(SPLADE_MODEL); docids=list(DMAP); qids=list(TEST_QMAP)
        D=model.encode_document([DMAP[d] for d in docids],convert_to_sparse_tensor=True,batch_size=32,show_progress_bar=True)
        Q=model.encode_query([TEST_QMAP[q] for q in qids],convert_to_sparse_tensor=True,batch_size=32,show_progress_bar=True)
        try:D=D.cpu()
        except Exception:pass
        try:Q=Q.cpu()
        except Exception:pass
        hits=semantic_search(Q,D,top_k=min(SPLADE_TOPK,len(docids)),score_function=dot_score,query_chunk_size=16,corpus_chunk_size=50000)
        splade_run={str(q):[(str(docids[int(h['corpus_id'])]),float(h['score'])) for h in qhits] for q,qhits in zip(qids,hits)}
        splade_path.write_text(json.dumps(splade_run),encoding='utf-8')
        del model,D,Q,hits
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    print('SPLADE queries:',len(splade_run),'depth',max(map(len,splade_run.values())))
else:
    print('RUN_SPLADE=False: controlled family will use BM25+dense only. For manuscript parity, rerun with RUN_SPLADE=True.')

## 9. MC-RRF / exact-collapse / StableRRF core

`Exact-collapse` here follows the implementation literally: only **identical full supplied document-ID sequences** have the same SHA-256 signature. Different-length prefix-consistent lists are not declared exact copies.

In [ ]:
from collections import defaultdict
import hashlib
def _normalize_weights(names, weights=None):
    w = {n: 1.0 for n in names} if weights is None else {n: float(weights.get(n, 1.0)) for n in names}
    if any(v < 0 for v in w.values()):
        raise ValueError("RRF-family methods require nonnegative weights.")
    return w


def rrf_from_prefixes(rankings, depths, k=60.0, weights=None):
    if float(k) < 0:
        raise ValueError("RRF k must be nonnegative.")
    names = list(rankings); weights = _normalize_weights(names, weights)
    scores = defaultdict(float)
    for n in names:
        pref = prefix_docs(rankings[n], depths[n])
        for r, d in enumerate(pref, 1):
            scores[d] += weights[n] / (float(k) + r)
    return sorted(scores.items(), key=lambda x: (-x[1], x[0]))


def ranking_signature(seq):
    docs = [str(x[0]) if isinstance(x, (tuple, list)) else str(x) for x in seq]
    return hashlib.sha256("\x1f".join(docs).encode("utf-8")).hexdigest()


def collapse_exact_sources(rankings, depths=None):
    groups = defaultdict(list)
    for n, seq in rankings.items():
        groups[ranking_signature(seq)].append(n)
    reps = {}; rep_depths = {}; rep_groups = {}
    for gi, members in enumerate(sorted(groups.values(), key=lambda xs: tuple(sorted(xs)))):
        rep = sorted(members)[0]
        key = f"G{gi:02d}:{rep}"
        reps[key] = rankings[rep]
        rep_groups[key] = list(sorted(members))
        if depths is not None:
            rep_depths[key] = max(int(depths[m]) for m in members)
    return reps, rep_depths if depths is not None else None, rep_groups


def canonical_family_name(source_name):
    # Exact-copy stress-test names inherit their original source family.
    base = str(source_name).split("__copy")[0]
    if base.startswith("bm25"):
        return "BM25-family"
    return base


def provenance_family_map(rankings):
    return {n: canonical_family_name(n) for n in rankings}


def collapse_exact_within_families(rankings, depths, family_map):
    """Collapse exact copies only when they also share declared provenance family."""
    keyed = defaultdict(list)
    for n, seq in rankings.items():
        fam = str(family_map[n])
        keyed[(fam, ranking_signature(seq))].append(n)
    reps={}; rep_depths={}; rep_groups={}; rep_family={}
    for gi, ((fam,_), members) in enumerate(sorted(keyed.items(), key=lambda kv:(kv[0][0],tuple(sorted(kv[1]))))):
        rep=sorted(members)[0]; key=f"F{gi:02d}:{rep}"
        reps[key]=rankings[rep]
        rep_depths[key]=max(int(depths[m]) for m in members)
        rep_groups[key]=list(sorted(members)); rep_family[key]=fam
    return reps, rep_depths, rep_groups, rep_family


def source_similarity_matrix(rankings, depth=100, p=0.9):
    names = list(rankings)
    docs = {n: prefix_docs(rankings[n], min(depth, len(rankings[n]))) for n in names}
    S = np.eye(len(names), dtype=float)
    for i in range(len(names)):
        for j in range(i+1, len(names)):
            d = min(depth, len(docs[names[i]]), len(docs[names[j]]))
            s = rbo_finite(docs[names[i]], docs[names[j]], p=p, depth=d)
            S[i,j] = S[j,i] = float(np.clip(s, 0.0, 1.0))
    return names, S


def redundancy_weights(rankings, depth=100, p=0.9, gamma=1.0):
    """Legacy global-soft baseline. It does NOT conserve provenance-family mass."""
    names, S = source_similarity_matrix(rankings, depth=depth, p=p)
    redundancy = np.maximum(S.sum(axis=1), 1e-12)
    raw = redundancy ** (-float(gamma))
    raw = raw * (len(raw) / raw.sum())
    return {n: float(raw[i]) for i,n in enumerate(names)}, S


def invariant_rrf_exact(rankings, depths, k=60.0):
    reps, rep_depths, groups = collapse_exact_sources(rankings, depths)
    fused = rrf_from_prefixes(reps, rep_depths, k=k)
    return fused, {"groups":groups, "weights":{n:1.0 for n in reps}}


def invariant_rrf_soft(rankings, depths, k=60.0, sim_depth=100, p=0.9, gamma=1.0):
    """Legacy global-soft comparator retained because v4 results used it."""
    reps, rep_depths, groups = collapse_exact_sources(rankings, depths)
    w, S = redundancy_weights(reps, depth=sim_depth, p=p, gamma=gamma)
    fused = rrf_from_prefixes(reps, rep_depths, k=k, weights=w)
    return fused, {"groups":groups, "weights":w, "similarity":S, "representatives":list(reps)}


def mass_conserving_weights(rankings, family_map, family_masses=None, within_family="equal", sim_depth=100, p=0.9):
    names=list(rankings)
    family_masses = dict(family_masses or {})
    fam_members=defaultdict(list)
    for n in names:
        if n not in family_map:
            raise KeyError(f"Missing family for source {n}")
        fam_members[str(family_map[n])].append(n)

    weights={}; family_summary={}
    for fam, members in sorted(fam_members.items()):
        mass=float(family_masses.get(fam,1.0))
        if mass < 0:
            raise ValueError("Family masses must be nonnegative")
        if within_family == "equal" or len(members)==1:
            shares=np.ones(len(members),dtype=float)
        elif within_family == "soft":
            sub={m:rankings[m] for m in members}
            _,S=source_similarity_matrix(sub,depth=sim_depth,p=p)
            redundancy=np.maximum(S.sum(axis=1),1e-12)
            shares=1.0/redundancy
        else:
            raise ValueError("within_family must be 'equal' or 'soft'")
        shares=shares/shares.sum() if shares.sum()>0 else np.ones(len(members))/len(members)
        for m,s in zip(members,shares):
            weights[m]=float(mass*s)
        family_summary[fam]={"mass":mass,"members":list(members),"shares":{m:float(s) for m,s in zip(members,shares)}}
    return weights, family_summary


def mc_rrf(rankings, depths, k=60.0, family_map=None, family_masses=None, within_family="equal", sim_depth=100, p=0.9):
    family_map = dict(family_map or provenance_family_map(rankings))
    reps, rep_depths, groups, rep_family = collapse_exact_within_families(rankings, depths, family_map)
    weights, fam_summary = mass_conserving_weights(
        reps, rep_family, family_masses=family_masses, within_family=within_family, sim_depth=sim_depth, p=p
    )
    fused=rrf_from_prefixes(reps,rep_depths,k=k,weights=weights)
    return fused,{"groups":groups,"rep_family":rep_family,"weights":weights,"families":fam_summary,"representatives":list(reps)}


def infer_families_from_rbo(rankings, depth=100, p=0.9, threshold=0.75):
    names,S=source_similarity_matrix(rankings,depth=depth,p=p)
    parent=list(range(len(names)))
    def find(x):
        while parent[x]!=x:
            parent[x]=parent[parent[x]]; x=parent[x]
        return x
    def union(a,b):
        ra,rb=find(a),find(b)
        if ra!=rb: parent[rb]=ra
    for i in range(len(names)):
        for j in range(i+1,len(names)):
            if S[i,j] >= float(threshold): union(i,j)
    roots={}
    fam_map={}
    for i,n in enumerate(names):
        r=find(i)
        if r not in roots: roots[r]=f"AutoFamily-{len(roots)+1}"
        fam_map[n]=roots[r]
    return fam_map,S


def auto_mc_rrf(rankings, depths, k=60.0, sim_depth=100, p=0.9, threshold=0.75):
    reps, rep_depths, groups = collapse_exact_sources(rankings, depths)
    fam_map,S=infer_families_from_rbo(reps,depth=sim_depth,p=p,threshold=threshold)
    weights,fam_summary=mass_conserving_weights(reps,fam_map,within_family="equal")
    fused=rrf_from_prefixes(reps,rep_depths,k=k,weights=weights)
    return fused,{"groups":groups,"family_map":fam_map,"families":fam_summary,"weights":weights,"similarity":S,"representatives":list(reps)}


def family_mass_fraction(meta, predicate):
    total=sum(float(v["mass"]) for v in meta["families"].values())
    selected=0.0
    for fam,info in meta["families"].items():
        if predicate(fam,info): selected+=float(info["mass"])
    return selected/total if total>0 else np.nan

In [ ]:
from dataclasses import dataclass
@dataclass(frozen=True)
class BoundRow:
    docid: str
    lb: float
    ub: float
    n_observed: int
    n_censored: int
    @property
    def width(self): return self.ub-self.lb


def censored_rrf_bounds(rankings, depths, k=60.0, weights=None, complete_flags=None):
    if float(k) < 0: raise ValueError("RRF k must be nonnegative.")
    names=list(rankings)
    if not names: return [],0.0,{"caps":{},"effective_depths":{},"complete_flags":{}}
    weights=_normalize_weights(names,weights)
    complete_flags={n:bool((complete_flags or {}).get(n,False)) for n in names}
    rank_maps={}; caps={}; effective_depths={}; candidates=set()
    for n in names:
        pref=prefix_docs(rankings[n],depths[n]); L=len(pref)
        effective_depths[n]=L; rank_maps[n]={d:r for r,d in enumerate(pref,1)}; candidates.update(pref)
        caps[n]=0.0 if (complete_flags[n] and L>=len(rankings[n])) else weights[n]/(float(k)+L+1.0)
    rows=[]
    for d in candidates:
        lb=0.0; ub=0.0; nobs=0
        for n in names:
            r=rank_maps[n].get(d)
            if r is not None:
                c=weights[n]/(float(k)+r); lb+=c; ub+=c; nobs+=1
            else:
                ub+=caps[n]
        rows.append(BoundRow(str(d),float(lb),float(ub),int(nobs),int(len(names)-nobs)))
    rows.sort(key=lambda x:(-x.lb,-x.ub,x.docid))
    unseen_ub=float(sum(caps.values()))
    meta={"caps":caps,"effective_depths":effective_depths,"weights":weights,"complete_flags":complete_flags,"k":float(k),"rank_maps":rank_maps}
    return rows,unseen_ub,meta


def certify_topk(bounds, unseen_ub, K=10, atol=1e-15):
    if K<=0: raise ValueError("K must be positive")
    b=sorted(bounds,key=lambda x:(-x.lb,-x.ub,x.docid))
    if len(b)<K:
        return {"certified_set":False,"certified_order":False,"certified_prefix_len":0,"topk":[],"ordered_prefix":[],"set_margin":-np.inf,"reason":"fewer_than_K_observed_candidates"}
    selected=b[:K]; outsiders=b[K:]
    boundary_lb=min(x.lb for x in selected)
    max_out=max([x.ub for x in outsiders]+[float(unseen_ub)])
    set_margin=boundary_lb-max_out
    set_cert=bool(set_margin>atol)
    ordered=[]
    for j,row in enumerate(b):
        rem=max([x.ub for x in b[j+1:]]+[float(unseen_ub)])
        if row.lb>rem+atol: ordered.append(row.docid)
        else: break
    return {"certified_set":set_cert,"certified_order":len(ordered)>=K,"certified_prefix_len":len(ordered),
            "topk":[x.docid for x in selected],"ordered_prefix":ordered,"set_margin":float(set_margin),
            "reason":"ordered_topk_certified" if len(ordered)>=K else ("topk_set_certified" if set_cert else "bounds_overlap")}


def certificate_holds(cert,target,K):
    if target=="set": return bool(cert["certified_set"])
    if target=="order": return bool(cert["certified_prefix_len"]>=K)
    raise ValueError("target must be set or order")


def result_run_from_bounds(bounds):
    b=sorted(bounds,key=lambda x:(-x.lb,-x.ub,x.docid))
    return [(x.docid,float(x.lb)) for x in b]

In [ ]:
def observed_cost(rankings,depths):
    reads=0; union=set()
    for n,seq in rankings.items():
        pref=prefix_docs(seq,depths[n]); reads+=len(pref); union.update(pref)
    return {'candidate_reads':int(reads),'unique_candidates':int(len(union)),'duplicate_reads':int(reads-len(union))}

def _critical_docs(bounds,K,n_outsiders=25):
    b=sorted(bounds,key=lambda x:(-x.lb,-x.ub,x.docid)); top=[x.docid for x in b[:K]]
    outside=sorted(b[K:],key=lambda x:(-x.ub,x.docid))[:n_outsiders]
    return list(dict.fromkeys(top+[x.docid for x in outside]))

def choose_ranker_to_deepen(rankings,depths,bounds,meta,K=10,step=10,max_depths=None,n_outsiders=25):
    max_depths=max_depths or {n:len(rankings[n]) for n in rankings}; critical=_critical_docs(bounds,K,n_outsiders); scores={}
    for n,seq in rankings.items():
        L=int(meta['effective_depths'][n]); maxL=min(len(seq),int(max_depths.get(n,len(seq))))
        if L>=maxL: continue
        nextL=min(L+int(step),maxL); w=float(meta['weights'][n]); old=float(meta['caps'][n])
        new=0.0 if (meta['complete_flags'].get(n,False) and nextL>=len(rankings[n])) else w/(float(meta['k'])+nextL+1.0)
        reduction=max(0.0,old-new); observed=set(meta['rank_maps'][n]); affected=1+sum(d not in observed for d in critical)
        scores[n]=reduction*affected
    if not scores:return None,scores
    return max(scores,key=lambda n:(scores[n],-int(meta['effective_depths'][n]),n)),scores

def stable_rrf_adaptive(rankings,K=10,k=60.0,weights=None,target='order',start_depth=10,step=10,max_depths=None,complete_flags=None,n_outsiders=25,max_iterations=10000):
    names=list(rankings); max_depths=max_depths or {n:len(rankings[n]) for n in names}
    depths={n:min(int(start_depth),int(max_depths.get(n,len(rankings[n]))),len(rankings[n])) for n in names}; history=[]
    for it in range(max_iterations):
        bounds,u,meta=censored_rrf_bounds(rankings,depths,k=k,weights=weights,complete_flags=complete_flags)
        cert=certify_topk(bounds,u,K=K); cost=observed_cost(rankings,depths)
        history.append({'iteration':it,'depths':dict(depths),'certified_set':cert['certified_set'],'certified_order':cert['certified_order'],'prefix':cert['certified_prefix_len'],**cost})
        if certificate_holds(cert,target,K):
            return {'certified':True,'certificate':cert,'depths':dict(depths),'history':history,'stop_reason':'certified',**cost}
        expandable=[n for n in names if depths[n]<min(len(rankings[n]),int(max_depths.get(n,len(rankings[n]))))]
        if not expandable:
            return {'certified':False,'certificate':cert,'depths':dict(depths),'history':history,'stop_reason':'budget_exhausted',**cost}
        chosen,_=choose_ranker_to_deepen(rankings,depths,bounds,meta,K=K,step=step,max_depths=max_depths,n_outsiders=n_outsiders)
        if chosen not in expandable:chosen=min(expandable,key=lambda n:(depths[n],n))
        depths[chosen]=min(depths[chosen]+int(step),len(rankings[chosen]),int(max_depths.get(chosen,len(rankings[chosen]))))
    raise RuntimeError('Adaptive deepening exceeded max_iterations')

## 10. Statistical helpers (same definitions as the v5 statistical pass)

In [ ]:
def bootstrap_mean_ci(values,n_boot=N_BOOT,seed=STAT_SEED,alpha=ALPHA,batch_size=500):
    x=np.asarray(values,dtype=float); x=x[np.isfinite(x)]; n=len(x)
    if n==0:return np.nan,np.nan,np.nan
    if n==1:return float(x[0]),float(x[0]),float(x[0])
    rng=np.random.default_rng(seed); boot=np.empty(n_boot); pos=0
    while pos<n_boot:
        b=min(batch_size,n_boot-pos); idx=rng.integers(0,n,size=(b,n)); boot[pos:pos+b]=x[idx].mean(axis=1); pos+=b
    return float(np.mean(x)),float(np.quantile(boot,alpha/2)),float(np.quantile(boot,1-alpha/2))

def wilcoxon_safe(values):
    x=np.asarray(values,dtype=float); x=x[np.isfinite(x)]
    if len(x)==0:return np.nan,np.nan
    if len(x[np.abs(x)>1e-15])==0:return 0.0,1.0
    with warnings.catch_warnings():
        warnings.simplefilter('ignore')
        try:
            r=wilcoxon(x,zero_method='wilcox',correction=False,alternative='two-sided',method='auto'); return float(r.statistic),float(r.pvalue)
        except Exception:return np.nan,np.nan

def paired_rank_biserial(values):
    x=np.asarray(values,dtype=float); x=x[np.isfinite(x)]; x=x[np.abs(x)>1e-15]
    if len(x)==0:return 0.0
    ranks=rankdata(np.abs(x),method='average'); wp=float(ranks[x>0].sum()); wn=float(ranks[x<0].sum()); den=wp+wn
    return float((wp-wn)/den) if den else 0.0

def holm_adjust(pvalues):
    p=np.asarray(pvalues,dtype=float); out=np.full(len(p),np.nan); valid=np.where(np.isfinite(p))[0]
    if len(valid)==0:return out
    pv=p[valid]; order=np.argsort(pv); sp=pv[order]; m=len(sp); adj_s=np.empty(m); running=0.0
    for i,raw in enumerate(sp):
        running=max(running,min(1.0,(m-i)*raw)); adj_s[i]=running
    adj=np.empty(m); adj[order]=adj_s; out[valid]=adj; return out

## 11. Exact-copy diagnostic — regenerated for SciFact

In [ ]:
base_names=['bm25'] + (['splade'] if splade_run is not None else []) + ['dense']
RUNS={'bm25':bm25_runs['bm25'],'bm25_lowb':bm25_runs['bm25_lowb'],'bm25_highb':bm25_runs['bm25_highb'],'dense':dense_run}
if splade_run is not None: RUNS['splade']=splade_run

rep_rows=[]
for attacked in base_names:
    for L in [20,50,100]:
        for copies in [1,2,4,8]:
            oo=[];oset=[];eo=[];mo=[]
            for q in test_qids:
                rankings={r:RUNS[r][q] for r in base_names}; depths={r:min(L,len(rankings[r])) for r in base_names}
                base_ord=[d for d,_ in rrf_from_prefixes(rankings,depths,k=RRF_K)[:EVAL_K]]
                base_ex=[d for d,_ in invariant_rrf_exact(rankings,depths,k=RRF_K)[0][:EVAL_K]]
                base_mc=[d for d,_ in mc_rrf(rankings,depths,k=RRF_K,family_map=provenance_family_map(rankings))[0][:EVAL_K]]
                dup=dict(rankings); dd=dict(depths); fm=provenance_family_map(rankings)
                for ci in range(1,copies):
                    name=f'{attacked}__copy{ci}'; dup[name]=rankings[attacked]; dd[name]=depths[attacked]; fm[name]=canonical_family_name(attacked)
                o=[d for d,_ in rrf_from_prefixes(dup,dd,k=RRF_K)[:EVAL_K]]
                e=[d for d,_ in invariant_rrf_exact(dup,dd,k=RRF_K)[0][:EVAL_K]]
                m=[d for d,_ in mc_rrf(dup,dd,k=RRF_K,family_map=fm)[0][:EVAL_K]]
                oo.append(int(o==base_ord)); oset.append(int(set(o)==set(base_ord))); eo.append(int(e==base_ex)); mo.append(int(m==base_mc))
            row={'dataset':'SciFact','attacked_source':attacked,'depth':L,'copies':copies,
                 'ordinary_exact_order_rate':np.mean(oo),'ordinary_exact_set_rate':np.mean(oset),
                 'exact_collapse_order_rate':np.mean(eo),'mc_rrf_order_rate':np.mean(mo)}
            assert row['exact_collapse_order_rate']==1.0 and row['mc_rrf_order_rate']==1.0
            rep_rows.append(row)
rep_df=pd.DataFrame(rep_rows); rep_df.to_csv(TABLE_DIR/'scifact_exact_replication.csv',index=False)
display(rep_df[(rep_df.depth==50)&(rep_df.copies==2)])
print('Mean ordinary ordered top-10 change after one copy at depth 50:',1-rep_df[(rep_df.depth==50)&(rep_df.copies==2)].ordinary_exact_order_rate.mean())

## 12. Controlled real BM25-family result + corrected global Holm family

In [ ]:
assert splade_run is not None, 'For manuscript-parity controlled BM25 family, set RUN_SPLADE=True and rerun.'
base=['bm25','splade','dense']; attacked=['bm25','bm25_lowb','bm25_highb','splade','dense']
family_query=[]; family_summary=[]
for L in [20,50,100]:
    ord_set=[];mc_set=[];ord_order=[];mc_order=[];ord_delta=[];mc_delta=[];mc_vs_ord=[];rbos=[]
    for q in test_qids:
        br={r:RUNS[r][q] for r in base}; bd={r:min(L,len(br[r])) for r in base}
        ar={r:RUNS[r][q] for r in attacked}; ad={r:min(L,len(ar[r])) for r in attacked}
        b=rrf_from_prefixes(br,bd,k=RRF_K); o=rrf_from_prefixes(ar,ad,k=RRF_K)
        m,_=mc_rrf(ar,ad,k=RRF_K,family_map=provenance_family_map(ar))
        bt=[d for d,_ in b[:10]]; ot=[d for d,_ in o[:10]]; mt=[d for d,_ in m[:10]]
        bn=ndcg_at_k(bt,qrels[q],10); on=ndcg_at_k(ot,qrels[q],10); mn=ndcg_at_k(mt,qrels[q],10)
        ord_set.append(int(set(ot)==set(bt))); mc_set.append(int(set(mt)==set(bt))); ord_order.append(int(ot==bt)); mc_order.append(int(mt==bt))
        ord_delta.append(on-bn); mc_delta.append(mn-bn); mc_vs_ord.append(mn-on)
        rbos.append(np.mean([rbo_finite(prefix_docs(ar[a],L),prefix_docs(ar[b],L),p=.9,depth=L) for a,b in [('bm25','bm25_lowb'),('bm25','bm25_highb'),('bm25_lowb','bm25_highb')]]))
        if L==50: family_query.append({'qid':q,'baseline_ndcg':bn,'ordinary_attacked_ndcg':on,'mc_attacked_ndcg':mn,'delta_ordinary':on-bn,'delta_mc':mn-bn,'mc_vs_ordinary':mn-on})
    family_summary.append({'dataset':'SciFact','depth':L,'mean_rbo':np.mean(rbos),'ordinary_set':np.mean(ord_set),'mc_set':np.mean(mc_set),'ordinary_order':np.mean(ord_order),'mc_order':np.mean(mc_order),'ordinary_delta':np.mean(ord_delta),'mc_delta':np.mean(mc_delta)})
family_summary_df=pd.DataFrame(family_summary); family_summary_df.to_csv(TABLE_DIR/'scifact_controlled_bm25_family.csv',index=False)

fq=pd.DataFrame(family_query)
# genuine new singleton SPLADE family: BM25+dense -> BM25+SPLADE+dense
new_delta=[]
for q in test_qids:
    before={'bm25':RUNS['bm25'][q],'dense':RUNS['dense'][q]}; after={**before,'splade':RUNS['splade'][q]}
    db={r:min(50,len(x)) for r,x in before.items()}; da={r:min(50,len(x)) for r,x in after.items()}
    bt=[d for d,_ in rrf_from_prefixes(before,db,k=RRF_K)[:10]]; at=[d for d,_ in rrf_from_prefixes(after,da,k=RRF_K)[:10]]
    new_delta.append(ndcg_at_k(at,qrels[q],10)-ndcg_at_k(bt,qrels[q],10))

new_tests=[]
for name,x,seedoff in [('ordinary_family_effect',fq.delta_ordinary.to_numpy(),0),('MC_vs_ordinary_attacked',fq.mc_vs_ordinary.to_numpy(),10000),('SPLADE_new_family_effect',np.array(new_delta),20000)]:
    mean,lo,hi=bootstrap_mean_ci(x,seed=STAT_SEED+seedoff+sum(map(ord,'SciFact')))
    stat,p=wilcoxon_safe(x)
    new_tests.append({'dataset':'SciFact','test':name,'mean_difference':mean,'ci_low':lo,'ci_high':hi,'p_raw':p,'rank_biserial':paired_rank_biserial(x)})

# Other five tests are unchanged because this rerun changes only SciFact's dense run.
other_tests=[
 {'dataset':'ArguAna','test':'ordinary_family_effect','p_raw':1.013975e-26},
 {'dataset':'ArguAna','test':'MC_vs_ordinary_attacked','p_raw':2.423865e-22},
 {'dataset':'FiQA','test':'ordinary_family_effect','p_raw':5.618559e-21},
 {'dataset':'FiQA','test':'MC_vs_ordinary_attacked','p_raw':2.120257e-24},
 {'dataset':'ArguAna','test':'SPLADE_new_family_effect','p_raw':1.877571e-13},
]
holm8=pd.DataFrame(other_tests+new_tests); holm8['p_holm']=holm_adjust(holm8.p_raw.to_numpy(float)); holm8['significant_0.05']=holm8.p_holm<.05
holm8.to_csv(STAT_DIR/'controlled_family_holm8_updated.csv',index=False)
display(family_summary_df[family_summary_df.depth==50]); display(holm8[holm8.dataset=='SciFact'])

## 13. Primary cross-source source-count scaling + corrected global Holm family

In [ ]:
def deterministic_seed(*parts):
    s='|'.join(map(str,parts)).encode('utf-8'); return int(hashlib.sha256(s).hexdigest()[:16],16)%(2**32-1)
def locally_perturb_ranking(seq,rate,seed,depth=100):
    docs=prefix_docs(seq,depth); n=len(docs)
    if n<2 or rate<=0:return [(d,0.0) for d in docs]
    rng=np.random.default_rng(seed); n_swaps=max(1,int(round(float(rate)*n))); positions=np.arange(n-1); n_swaps=min(n_swaps,len(positions))
    chosen=sorted(rng.choice(positions,size=n_swaps,replace=False).tolist()); out=list(docs); last=-2
    for p in chosen:
        if p==last+1:continue
        out[p],out[p+1]=out[p+1],out[p]; last=p
    assert set(out)==set(docs); assert out!=docs or n<2
    return [(d,0.0) for d in out]
def make_redundant_family(base_seq,family_size,rate,key):
    fam={'original':base_seq}
    for j in range(1,int(family_size)): fam[f'variant{j}']=locally_perturb_ranking(base_seq,rate,deterministic_seed(*key,j),depth=SOURCE_DEPTH)
    return fam
def rrf_from_rankings(rankings,depth=100,k=60.,weights=None):
    return rrf_from_prefixes(rankings,{n:min(depth,len(s)) for n,s in rankings.items()},k=k,weights=weights)
def mc_family_weights(fam_names,other):
    m=len(fam_names); w={n:1.0/m for n in fam_names}; w[other]=1.0; return w

primary_rows=[]; primary_perq=[]
base_runs={'bm25':RUNS['bm25'],'dense':RUNS['dense']}
for attacked in ['bm25','dense']:
    other='dense' if attacked=='bm25' else 'bm25'
    os=[];ms=[];oo=[];mo=[];osig=[];msig=[];oabs=[];mabs=[];rbos=[]
    for q in test_qids:
        ba=base_runs[attacked][q]; bo=base_runs[other][q]
        clean=rrf_from_rankings({attacked:ba,other:bo},depth=100,k=60); ct=[d for d,_ in clean[:10]]; cn=ndcg_at_k(ct,qrels[q],10)
        fam=make_redundant_family(ba,8,.05,('SciFact',q,attacked,'mild'))
        expanded={f'{attacked}__{name}':seq for name,seq in fam.items()}; expanded[other]=bo
        fam_names=[n for n in expanded if n.startswith(attacked+'__')]
        ordinary=rrf_from_rankings(expanded,depth=100,k=60); mc=rrf_from_rankings(expanded,depth=100,k=60,weights=mc_family_weights(fam_names,other))
        ot=[d for d,_ in ordinary[:10]]; mt=[d for d,_ in mc[:10]]; on=ndcg_at_k(ot,qrels[q],10); mn=ndcg_at_k(mt,qrels[q],10)
        os.append(int(set(ot)==set(ct))); ms.append(int(set(mt)==set(ct))); oo.append(int(ot==ct)); mo.append(int(mt==ct))
        osig.append(on-cn); msig.append(mn-cn); oabs.append(abs(on-cn)); mabs.append(abs(mn-cn))
        parent=prefix_docs(ba,100)
        for n,s in fam.items():
            if n!='original':rbos.append(rbo_finite(parent,prefix_docs(s,100),p=.9,depth=100))
        primary_perq.append({'dataset':'SciFact','attacked_source':attacked,'qid':q,'ordinary_abs_drift':abs(on-cn),'mc_abs_drift':abs(mn-cn),'ordinary_signed_delta':on-cn,'mc_signed_delta':mn-cn,'ordinary_set_preserved':int(set(ot)==set(ct)),'mc_set_preserved':int(set(mt)==set(ct))})
    dif=np.asarray(mabs)-np.asarray(oabs); mean,lo,hi=bootstrap_mean_ci(dif,n_boot=10000,seed=41); _,p=wilcoxon_safe(dif)
    primary_rows.append({'dataset':'SciFact','attacked_source':attacked,'mean_family_rbo':np.mean(rbos),'ordinary_set_preservation':np.mean(os),'mc_set_preservation':np.mean(ms),'ordinary_order_preservation':np.mean(oo),'mc_order_preservation':np.mean(mo),'ordinary_signed_ndcg_delta':np.mean(osig),'mc_signed_ndcg_delta':np.mean(msig),'ordinary_abs_ndcg_drift':np.mean(oabs),'mc_abs_ndcg_drift':np.mean(mabs),'mean_mc_minus_ordinary_abs_drift':mean,'ci_low':lo,'ci_high':hi,'p_raw':p,'rank_biserial':paired_rank_biserial(dif)})
primary_df=pd.DataFrame(primary_rows)

# Preserve the six non-SciFact raw p-values from the already executed scaling notebook and recompute Holm over the complete 8-test family.
other_scaling=[
 ('ArguAna','bm25',2.100331e-111),('ArguAna','dense',9.284136e-109),('FiQA','bm25',2.414466e-51),('FiQA','dense',6.335628e-57),('TREC-COVID','bm25',1.256513e-09),('TREC-COVID','dense',2.396832e-09)]
global_p=pd.DataFrame([{'dataset':d,'attacked_source':s,'p_raw':p} for d,s,p in other_scaling]+primary_df[['dataset','attacked_source','p_raw']].to_dict('records'))
global_p['p_holm']=holm_adjust(global_p.p_raw.to_numpy(float)); global_p['significant_0.05']=global_p.p_holm<.05
primary_df=primary_df.merge(global_p[['dataset','attacked_source','p_holm','significant_0.05']],on=['dataset','attacked_source'],how='left')
primary_df['drift_reduction']=primary_df.ordinary_abs_ndcg_drift-primary_df.mc_abs_ndcg_drift
primary_df['set_preservation_gain']=primary_df.mc_set_preservation-primary_df.ordinary_set_preservation
primary_df.to_csv(TABLE_DIR/'scifact_cross_source_primary_updated.csv',index=False); global_p.to_csv(STAT_DIR/'cross_source_holm8_updated.csv',index=False)
display(primary_df)

## 14. SciFact sensitivity checks: perturbation geometry, `k × K`, provenance splitting

In [ ]:
if RUN_SENSITIVITY:
    def perturb_adjacent(seq,rate,seed,depth=100): return locally_perturb_ranking(seq,rate,seed,depth)
    def perturb_bounded_displacement(seq,rate,seed,depth=100,max_disp=5):
        docs=prefix_docs(seq,depth); n=len(docs)
        if n<2 or rate<=0:return [(d,0.0) for d in docs]
        rng=np.random.default_rng(seed); out=list(docs); n_moves=max(1,int(round(rate*n)))
        # Source-truth detail: process selected documents in the exact order returned by NumPy RNG (not sorted).
        selected=rng.choice(np.arange(n),size=min(n_moves,n),replace=False)
        for original_pos in selected:
            cur=out.index(docs[int(original_pos)]); disp=0
            while disp==0:disp=int(rng.integers(-max_disp,max_disp+1))
            new=max(0,min(n-1,cur+disp)); item=out.pop(cur); out.insert(new,item)
        if out==docs:out[0],out[1]=out[1],out[0]
        assert set(out)==set(docs); return [(d,0.0) for d in out]
    def perturb_block_permutation(seq,rate,seed,depth=100,block_size=5):
        docs=prefix_docs(seq,depth); n=len(docs)
        if n<2 or rate<=0:return [(d,0.0) for d in docs]
        rng=np.random.default_rng(seed); out=list(docs); blocks=[(s,min(n,s+block_size)) for s in range(0,n,block_size) if min(n,s+block_size)-s>=2]
        n_blocks=max(1,int(round(rate*n/max(2,block_size)))); chosen=rng.choice(np.arange(len(blocks)),size=min(n_blocks,len(blocks)),replace=False)
        for bi in chosen:
            s,e=blocks[int(bi)]; block=out[s:e]; perm=list(rng.permutation(block));
            if perm==block:perm=block[1:]+block[:1]
            out[s:e]=perm
        if out==docs:out[0],out[1]=out[1],out[0]
        assert set(out)==set(docs); return [(d,0.0) for d in out]
    PF={'adjacent_swap':perturb_adjacent,'bounded_displacement':perturb_bounded_displacement,'block_permutation':perturb_block_permutation}
    def make_family(base_seq,size,rate,mechanism,key):
        fam={'original':base_seq}; fn=PF[mechanism]
        for j in range(1,size):fam[f'variant{j}']=fn(base_seq,rate,deterministic_seed(*key,mechanism,j),depth=100)
        return fam

    sens=[]
    for attacked in ['bm25','dense']:
        other='dense' if attacked=='bm25' else 'bm25'
        for mech in PF:
            oa=[];ma=[];os=[];ms=[]
            for q in test_qids:
                ba=base_runs[attacked][q];bo=base_runs[other][q]; clean=rrf_from_rankings({attacked:ba,other:bo},100,60);ct=[d for d,_ in clean[:10]];cn=ndcg_at_k(ct,qrels[q],10)
                fam=make_family(ba,8,.05,mech,('SciFact',q,attacked,'mild')); ex={f'{attacked}__{n}':s for n,s in fam.items()};ex[other]=bo;names=[n for n in ex if n.startswith(attacked+'__')]
                o=rrf_from_rankings(ex,100,60);m=rrf_from_rankings(ex,100,60,mc_family_weights(names,other));ot=[d for d,_ in o[:10]];mt=[d for d,_ in m[:10]]
                oa.append(abs(ndcg_at_k(ot,qrels[q],10)-cn));ma.append(abs(ndcg_at_k(mt,qrels[q],10)-cn));os.append(int(set(ot)==set(ct)));ms.append(int(set(mt)==set(ct)))
            diff=np.asarray(ma)-np.asarray(oa);_,p=wilcoxon_safe(diff)
            sens.append({'attacked_source':attacked,'mechanism':mech,'ordinary_abs_drift':np.mean(oa),'mc_abs_drift':np.mean(ma),'ordinary_set':np.mean(os),'mc_set':np.mean(ms),'p_raw_scifact_only':p})
    pd.DataFrame(sens).to_csv(TABLE_DIR/'scifact_perturbation_mechanisms.csv',index=False)

    kk=[]
    for kval in [10,30,60,100]:
      for K in [5,10,20]:
       for attacked in ['bm25','dense']:
        other='dense' if attacked=='bm25' else 'bm25';oa=[];ma=[];os=[];ms=[]
        for q in test_qids:
            ba=base_runs[attacked][q];bo=base_runs[other][q];clean=rrf_from_rankings({attacked:ba,other:bo},100,kval);ct=[d for d,_ in clean[:K]];cn=ndcg_at_k(ct,qrels[q],10)
            fam=make_redundant_family(ba,8,.05,('SciFact',q,attacked,'mild'));ex={f'{attacked}__{n}':s for n,s in fam.items()};ex[other]=bo;names=[n for n in ex if n.startswith(attacked+'__')]
            o=rrf_from_rankings(ex,100,kval);m=rrf_from_rankings(ex,100,kval,mc_family_weights(names,other));ot=[d for d,_ in o[:K]];mt=[d for d,_ in m[:K]]
            oa.append(abs(ndcg_at_k(ot,qrels[q],10)-cn));ma.append(abs(ndcg_at_k(mt,qrels[q],10)-cn));os.append(int(set(ot)==set(ct)));ms.append(int(set(mt)==set(ct)))
        kk.append({'k':kval,'K':K,'attacked_source':attacked,'ordinary_abs_drift':np.mean(oa),'mc_abs_drift':np.mean(ma),'ordinary_set':np.mean(os),'mc_set':np.mean(ms)})
    pd.DataFrame(kk).to_csv(TABLE_DIR/'scifact_k_by_K.csv',index=False)

    # Under-grouping curve, matching the limitation notebook's deterministic split of sorted variant names.
    prov=[]
    for attacked in ['bm25','dense']:
        other='dense' if attacked=='bm25' else 'bm25'
        for split_rate in [0,.25,.50,.75,1.0]:
            sp=[];dr=[];mass=[]
            for q in test_qids:
                ba=base_runs[attacked][q];bo=base_runs[other][q];clean=rrf_from_rankings({attacked:ba,other:bo},100,60);ct=[d for d,_ in clean[:10]];cn=ndcg_at_k(ct,qrels[q],10)
                fam=make_family(ba,8,.05,'adjacent_swap',('SciFact',q,attacked,'prov'));ex={f'{attacked}__{n}':s for n,s in fam.items()};ex[other]=bo;fnames=sorted([n for n in ex if n.startswith(attacked+'__')]);original=[n for n in fnames if n.endswith('__original')];variants=[n for n in fnames if n not in original]
                n_split=int(round(split_rate*len(variants))); split=set(variants[:n_split]);groups={other:'OTHER'}
                for n in fnames:groups[n]=f'SPLIT::{n}' if n in split else 'ATTACKED_CORE'
                gs=defaultdict(list)
                for s,g in groups.items():gs[g].append(s)
                w={};
                for g,mem in gs.items():
                    for s in mem:w[s]=1.0/len(mem)
                fused=rrf_from_rankings(ex,100,60,w);top=[d for d,_ in fused[:10]];sp.append(int(set(top)==set(ct)));dr.append(abs(ndcg_at_k(top,qrels[q],10)-cn));mass.append(sum(w[n] for n in fnames)/sum(w.values()))
            prov.append({'attacked_source':attacked,'split_rate':split_rate,'attacked_mass':np.mean(mass),'set_preservation':np.mean(sp),'abs_drift':np.mean(dr)})
    pd.DataFrame(prov).to_csv(TABLE_DIR/'scifact_provenance_splitting.csv',index=False)
    print('Sensitivity outputs saved.')

## 15. StableRRF with the leakage-clean dense run

In [ ]:
if RUN_STABLERFF:
    stable_rows=[]
    source_sets=[('BM25+Dense',['bm25','dense'])]
    if splade_run is not None: source_sets.append(('BM25+SPLADE+Dense',['bm25','splade','dense']))
    for label,sources in source_sets:
        qids=sorted(set(test_qids).intersection(*(set(RUNS[s]) for s in sources)))
        max_depths={'bm25':1000,'dense':100,'splade':500}
        for target in ['order','set']:
            cert=[];reads=[]
            for q in qids:
                rankings={s:RUNS[s][q] for s in sources}; mdp={s:min(max_depths[s],len(rankings[s])) for s in sources}
                res=stable_rrf_adaptive(rankings,K=10,k=60,target=target,start_depth=10,step=10,max_depths=mdp,n_outsiders=25)
                cert.append(int(res['certified'])); reads.append(res['candidate_reads'])
            budget=sum(min(max_depths[s],max(len(RUNS[s][q]) for q in qids)) for s in sources)
            stable_rows.append({'dataset':'SciFact','source_set':label,'target':target,'cert_rate':np.mean(cert),'mean_candidate_reads':np.mean(reads),'deepest_observed_budget':budget,'saving_pct':100*(1-np.mean(reads)/budget)})
    stable_df=pd.DataFrame(stable_rows);stable_df.to_csv(TABLE_DIR/'scifact_stablerrf_updated.csv',index=False);display(stable_df)

    # Static depth-50 certification for K=1 and K=10, separate from adaptive schedule.
    static=[]
    for K in [1,10]:
        vals=[]
        for q in test_qids:
            rankings={'bm25':RUNS['bm25'][q],'dense':RUNS['dense'][q]}; depths={s:min(50,len(r)) for s,r in rankings.items()}
            b,u,_=censored_rrf_bounds(rankings,depths,k=60); c=certify_topk(b,u,K=K); vals.append(int(c['certified_prefix_len']>=K))
        static.append({'K':K,'depth':50,'ordered_cert_rate':np.mean(vals)})
    pd.DataFrame(static).to_csv(TABLE_DIR/'scifact_static_depth50_certification.csv',index=False); display(pd.DataFrame(static))

## 16. Final integrity checks, comparison to old SciFact values, and upload package

In [ ]:
# Fail loudly on the core repair and strict checkpoint reload.
assert json.loads(LOCK_PATH.read_text())['selection_policy'].startswith('fixed final epoch')
assert not (TRAIN_ROOT/'dev'/'qrels.tsv').exists()
assert sha256_file(FINAL_MODEL/'model.pt')==model_hash
assert set(train_qids).isdisjoint(test_qids)
# The dense run can exist only after load_fixed_model() completed strict=True audits.
assert (RUN_DIR/'run_dense_clean_seed41_epoch3_STRICT.json').exists()
assert (rep_df.exact_collapse_order_rate==1.0).all() and (rep_df.mc_rrf_order_rate==1.0).all()
assert (primary_df.mc_abs_ndcg_drift < primary_df.ordinary_abs_ndcg_drift).all(), 'MC no longer reduces SciFact primary drift in both attacks; report honestly.'
assert (primary_df.mc_set_preservation > primary_df.ordinary_set_preservation).all(), 'MC no longer raises SciFact set preservation in both attacks; report honestly.'

old_scaling=pd.DataFrame([
 {'attacked_source':'bm25','old_ord_set':.003333,'old_mc_set':.986667,'old_ord_abs':.073244,'old_mc_abs':.007548,'old_signed_ord':-.026295,'old_signed_mc':.000999},
 {'attacked_source':'dense','old_ord_set':.013333,'old_mc_set':.983333,'old_ord_abs':.081265,'old_mc_abs':.005282,'old_signed_ord':-.006299,'old_signed_mc':-.001076},
])
comparison=old_scaling.merge(primary_df,on='attacked_source'); comparison.to_csv(TABLE_DIR/'scifact_old_vs_clean_primary_scaling.csv',index=False)

lines=[]
lines += ['# SciFact leakage-clean rerun — paper-facing results','',f'- Model: `{MODEL_NAME}`',f'- Seed: {SEED}',f'- Fixed epochs: {EPOCHS}',f'- Model SHA-256: `{model_hash}`','- Checkpoint selection: **none**; fixed final epoch, test qrels loaded only after model lock.','']
lines += ['## Baseline dense/BM25',f'- Dense nDCG@10: {baseline_dense:.6f}',f'- BM25 nDCG@10: {baseline_bm25:.6f}','']
lines += ['## Exact-copy diagnostic (depth 50, one additional copy)']
r=rep_df[(rep_df.depth==50)&(rep_df.copies==2)]
for x in r.itertuples(): lines.append(f'- {x.attacked_source}: ordinary order preservation={x.ordinary_exact_order_rate:.4f}; exact-collapse={x.exact_collapse_order_rate:.4f}; MC-RRF={x.mc_rrf_order_rate:.4f}')
lines += ['','## Controlled BM25 family (depth 50)']
for x in family_summary_df[family_summary_df.depth==50].itertuples(): lines.append(f'- RBO={x.mean_rbo:.4f}; ordinary set={x.ordinary_set:.4f}; MC set={x.mc_set:.4f}; ordinary ΔnDCG={x.ordinary_delta:+.6f}; MC ΔnDCG={x.mc_delta:+.6f}')
for x in holm8[holm8.dataset=='SciFact'].itertuples(): lines.append(f'- {x.test}: mean={getattr(x,"mean_difference",float("nan")):+.6f}; Holm p={x.p_holm:.6g}')
lines += ['','## Primary cross-source scaling (m=8, 5%, depth=100, k=60, K=10)']
for x in primary_df.itertuples(): lines.append(f'- attack {x.attacked_source}: ordinary set={x.ordinary_set_preservation:.4f}; MC set={x.mc_set_preservation:.4f}; ordinary abs drift={x.ordinary_abs_ndcg_drift:.6f}; MC abs drift={x.mc_abs_ndcg_drift:.6f}; reduction={x.drift_reduction:.6f}; Holm p={x.p_holm:.6g}; signed ordinary={x.ordinary_signed_ndcg_delta:+.6f}; signed MC={x.mc_signed_ndcg_delta:+.6f}')
if RUN_STABLERFF:
    lines += ['','## StableRRF']
    for x in stable_df.itertuples(): lines.append(f'- {x.source_set} / {x.target}: cert={x.cert_rate:.4f}; reads={x.mean_candidate_reads:.2f}; saving={x.saving_pct:.2f}%')
lines += ['','## Interpretation guard','Use these results to replace **only SciFact-dependent rows/numbers** in the manuscript. Non-SciFact results remain from the previously executed notebooks. Recompute macro summaries after inserting the new SciFact rows.']
report='\n'.join(lines); (OUT/'SCIFACT_RESULTS.md').write_text(report,encoding='utf-8'); print(report)

manifest={'model_lock':json.loads(LOCK_PATH.read_text()),'files':{}}
for p in sorted(OUT.rglob('*')):
    if p.is_file() and p.name!='model.pt':
        try: manifest['files'][str(p.relative_to(OUT))]=sha256_file(p)
        except Exception: pass
(OUT/'RUN_MANIFEST.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')

zip_path=ROOT/'InvariantRRF_SciFact_Leakage_Clean_Results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(OUT.rglob('*')):
        if not p.is_file():continue
        # Do not package the 400+ MB model weights; the hash is in MODEL_LOCK.json.
        if p.name=='model.pt':continue
        z.write(p,arcname=str(p.relative_to(OUT)))
print(OUT/'SCIFACT_RESULTS.md')
print(zip_path)

## 17. Canonical four-dataset strict rerun

The SciFact-only repair above establishes a leakage-clean, fixed epoch-3 checkpoint and verifies strict loading. The remaining cells make that **same locked checkpoint** the canonical dense source for all four paper datasets.

They reuse the exact MPDR query/document/qrel populations from the earlier InvariantRRF experiments: SciFact=300, TREC-COVID=50, FiQA=648, and ArguAna=1401. Historical `run_dense.json` files are not reused.

The canonical pass regenerates:

- strict dense top-100 rankings for all four datasets;
- BM25 top-1000 and the real BM25 variants;
- exact-copy diagnostics with method-to-own-baseline invariance checks;
- the real BM25-family experiment and its 8-test Holm family;
- controlled source-count scaling at family sizes 1, 2, 4, and 8 under mild and moderate perturbations;
- the 8-test primary scaling Holm family (mild, m=8);
- the original 24-test perturbation-mechanism protocol;
- the original 12-cell `k x K` protocol, using the dedicated `kK` seed key and nDCG at the requested K;
- provenance under-grouping and over-grouping;
- StableRRF set/order certification, static K sensitivity, the original 16-test scheduler audit, and a two-source finite-lattice oracle audit.

Run from the top of the notebook. Do not combine results from a partially executed run with the paper.

In [ ]:
# Canonical output root
CANON = ROOT / 'invariantrrf_canonical_strict_full'
CANON_RUNS = CANON / 'runs'
CANON_TABLES = CANON / 'tables'
CANON_STATS = CANON / 'statistics'
for p in [CANON, CANON_RUNS, CANON_TABLES, CANON_STATS]:
    p.mkdir(parents=True, exist_ok=True)

# The strict model produced above is the only dense checkpoint allowed here.
assert LOCK_PATH.exists()
_locked = json.loads(LOCK_PATH.read_text())
assert sha256_file(FINAL_MODEL/'model.pt') == _locked['model_sha256']

# Copy the model-selection lock into the canonical package. Model weights remain excluded.
(CANON/'MODEL_LOCK.json').write_text(json.dumps(_locked, indent=2), encoding='utf-8')
canonical_protocol = {
    'model_name': MODEL_NAME,
    'model_sha256': _locked['model_sha256'],
    'dense_checkpoint_policy': 'fixed final epoch 3; no validation/test checkpoint selection; strict=True reload',
    'dense_training_source': 'official BEIR SciFact train qrels only',
    'datasets': ['SciFact','TREC-COVID','FiQA','ArguAna'],
    'expected_queries': {'SciFact':300,'TREC-COVID':50,'FiQA':648,'ArguAna':1401},
    'dense_topk_generated': DENSE_TOPK_DEEP,
    'dense_primary_cap': STABLE_DENSE_CAP_PRIMARY,
    'stable_dense_cap_sensitivity': STABLE_DENSE_CAPS,
    'task_k_grid': TASK_K_GRID,
    'task_k_calibration_fraction': TASK_K_CALIB_FRACTION,
    'bm25_topk': 1000,
    'splade_checkpoint': 'naver/splade-cocondenser-ensembledistil',
    'source_depth_primary': 100,
    'rrf_k_primary': 60,
    'eval_k_primary': 10,
    'family_sizes': [1,2,4,8],
    'scaling_perturbations': {'mild':0.05,'moderate':0.15},
    'primary_scaling_condition': {'perturbation':'mild','family_size':8,'mechanism':'adjacent_swap'},
    'bootstrap_resamples': N_BOOT,
}
(CANON/'CANONICAL_PROTOCOL.json').write_text(json.dumps(canonical_protocol, indent=2), encoding='utf-8')


def read_jsonl_local(path):
    rows=[]
    with open(path,'r',encoding='utf-8') as f:
        for line in f:
            if line.strip(): rows.append(json.loads(line))
    return rows


def load_qrels_graded(root):
    qrels=defaultdict(dict)
    p=Path(root)/'dev'/'qrels.tsv'
    with open(p,'r',encoding='utf-8') as f:
        for line in f:
            sp=line.strip().split('\t')
            if len(sp)<2: continue
            qid,did=str(sp[0]),str(sp[1])
            rel=float(sp[2]) if len(sp)>=3 else 1.0
            if rel>0: qrels[qid][did]=rel
    return dict(qrels)


def discover_prev_root():
    # GitHub/public version: discover the attached benchmark bundle recursively.
    # Expected layout somewhere below the selected input root:
    #   mpdr/data/scifact_mpdr/dev/queries.jsonl
    base=Path('/kaggle/input')
    if base.exists():
        hits=list(base.rglob('scifact_mpdr/dev/queries.jsonl'))
        if hits:
            return hits[0].parents[4]
    raise FileNotFoundError('Attach the same prior Kaggle notebook/dataset input used by InvariantRRF v5.')

PREV=discover_prev_root()
OLD_DATA=PREV/'mpdr'/'data'
DATASET_ROOTS={
    'SciFact': OLD_DATA/'scifact_mpdr',
    'TREC-COVID': OLD_DATA/'trec-covid_mpdr',
    'FiQA': OLD_DATA/'fiqa_mpdr',
    'ArguAna': OLD_DATA/'arguana_mpdr',
}
EXPECTED_Q={'SciFact':300,'TREC-COVID':50,'FiQA':648,'ArguAna':1401}
print('Using prior benchmark root:',PREV)
for ds,p in DATASET_ROOTS.items():
    assert (p/'dev/queries.jsonl').exists(), (ds,p)
    assert (p/'dev/docs.jsonl').exists(), (ds,p)
    assert (p/'dev/qrels.tsv').exists(), (ds,p)
    print(ds,p)

# Strict model is loaded once with the exact training architecture.
STRICT_ENC, STRICT_TOK = load_fixed_model(FINAL_MODEL)
STRICT_DEVICE=torch.device('cuda' if torch.cuda.is_available() else 'cpu')
STRICT_ENC.to(STRICT_DEVICE).eval()
QPREF, DPREF=get_prefixes('e5')

@torch.no_grad()
def rank_dense_streaming(qmap,dmap,top_k=100,doc_block=2048,batch=64):
    # Exact dot-product top-k with bounded memory; uses the strictly loaded model.
    qids=list(qmap); docids=list(dmap)
    Q=embed_batches(STRICT_ENC,STRICT_TOK,[QPREF+qmap[q] for q in qids],STRICT_DEVICE,batch=batch,max_length=MAX_LENGTH).to(STRICT_DEVICE)
    k=min(int(top_k),len(docids))
    best_vals=torch.full((len(qids),k),-float('inf'),device=STRICT_DEVICE)
    best_idx=torch.full((len(qids),k),-1,dtype=torch.long,device=STRICT_DEVICE)
    for start in range(0,len(docids),doc_block):
        block_ids=docids[start:start+doc_block]
        D=embed_batches(STRICT_ENC,STRICT_TOK,[DPREF+dmap[d] for d in block_ids],STRICT_DEVICE,batch=batch,max_length=MAX_LENGTH).to(STRICT_DEVICE)
        scores=Q @ D.t()
        lk=min(k,scores.shape[1])
        lv,li=torch.topk(scores,k=lk,dim=1)
        li=li+start
        cand_v=torch.cat([best_vals,lv],dim=1)
        cand_i=torch.cat([best_idx,li],dim=1)
        best_vals,pos=torch.topk(cand_v,k=k,dim=1)
        best_idx=torch.gather(cand_i,1,pos)
        del D,scores,lv,li,cand_v,cand_i,pos
        if torch.cuda.is_available(): torch.cuda.empty_cache()
    vals=best_vals.detach().cpu().numpy(); idx=best_idx.detach().cpu().numpy()
    run={}
    for qi,q in enumerate(qids):
        run[str(q)]=[(str(docids[int(j)]),float(vals[qi,ri])) for ri,j in enumerate(idx[qi]) if int(j)>=0]
    del Q,best_vals,best_idx
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return run


def generate_splade_generic(qmap,dmap,top_k=500):
    from sentence_transformers import SparseEncoder
    from sentence_transformers.util import semantic_search,dot_score
    model=SparseEncoder('naver/splade-cocondenser-ensembledistil')
    docids=list(dmap); qids=list(qmap)
    D=model.encode_document([dmap[d] for d in docids],convert_to_sparse_tensor=True,batch_size=32,show_progress_bar=True)
    Q=model.encode_query([qmap[q] for q in qids],convert_to_sparse_tensor=True,batch_size=32,show_progress_bar=True)
    try: D=D.cpu()
    except Exception: pass
    try: Q=Q.cpu()
    except Exception: pass
    hits=semantic_search(Q,D,top_k=min(top_k,len(docids)),score_function=dot_score,query_chunk_size=16,corpus_chunk_size=50000)
    run={str(q):[(str(docids[int(h['corpus_id'])]),float(h['score'])) for h in qhits] for q,qhits in zip(qids,hits)}
    del model,D,Q,hits
    if torch.cuda.is_available(): torch.cuda.empty_cache()
    return run

# Build all four canonical dataset instruments.
DS_CANON={}
for ds_name,root in DATASET_ROOTS.items():
    if ds_name=='SciFact':
        # Use the already regenerated strict SciFact instruments from the preceding cells.
        qmap_ds=dict(TEST_QMAP); dmap_ds=dict(DMAP); qrels_ds=dict(qrels); qids_ds=list(test_qids)
        print('[SciFact] regenerating strict dense top-1000 from the locked checkpoint')
        dense_ds=rank_dense_streaming({q:qmap_ds[q] for q in qids_ds},dmap_ds,top_k=DENSE_TOPK_DEEP)
        bm25_family_ds=bm25_runs; splade_ds=splade_run
        # The historical strict top-100 SciFact run must be the exact prefix of the deeper run.
        prefix_mismatch=sum(prefix_docs(dense_ds[q],DENSE_TOPK)!=prefix_docs(dense_run[q],DENSE_TOPK) for q in qids_ds)
        assert prefix_mismatch==0, f'SciFact deep dense top-100 prefix mismatch on {prefix_mismatch} queries.'

        # IMPORTANT: also copy SciFact instruments into the canonical package.
        (CANON_RUNS/'SciFact_dense_STRICT_top1000.json').write_text(json.dumps(dense_ds),encoding='utf-8')
        (CANON_RUNS/'SciFact_dense_STRICT_top100.json').write_text(json.dumps(dense_run),encoding='utf-8')
        for nm,rr in bm25_family_ds.items():
            (CANON_RUNS/f'SciFact_{nm}_top1000.json').write_text(json.dumps(rr),encoding='utf-8')
        if splade_ds is not None:
            (CANON_RUNS/'SciFact_splade_ensemble_top500.json').write_text(json.dumps(splade_ds),encoding='utf-8')
    else:
        qs=read_jsonl_local(root/'dev/queries.jsonl'); docs_ds=read_jsonl_local(root/'dev/docs.jsonl')
        qmap_ds={str(x['id']):x.get('text','') for x in qs}; dmap_ds={str(x['id']):x.get('text','') for x in docs_ds}
        qrels_ds=load_qrels_graded(root); qids_ds=sorted(set(qrels_ds)&set(qmap_ds))
        assert len(qids_ds)==EXPECTED_Q[ds_name], (ds_name,len(qids_ds))
        print(f'[{ds_name}] strict dense ranking: {len(qids_ds)} queries / {len(dmap_ds)} docs')
        dense_ds=rank_dense_streaming({q:qmap_ds[q] for q in qids_ds},dmap_ds,top_k=DENSE_TOPK_DEEP)
        (CANON_RUNS/f'{ds_name}_dense_STRICT_top1000.json').write_text(json.dumps(dense_ds),encoding='utf-8')
        print(f'[{ds_name}] BM25 family')
        bm25_family_ds=generate_bm25_family({q:qmap_ds[q] for q in qids_ds},dmap_ds,top_k=1000)
        for nm,rr in bm25_family_ds.items():
            (CANON_RUNS/f'{ds_name}_{nm}_top1000.json').write_text(json.dumps(rr),encoding='utf-8')
        splade_ds=None
        if ds_name=='ArguAna':
            # Only reuse this notebook's explicit EnsembleDistil canonical cache.
            sp_cache=CANON_RUNS/'ArguAna_splade_ensemble_top500.json'
            if sp_cache.exists():
                candidate=json.loads(sp_cache.read_text())
                if set(qids_ds).issubset(candidate) and min(len(candidate[q]) for q in qids_ds) >= min(500,len(dmap_ds)):
                    splade_ds=candidate
                    print('[ArguAna] reused canonical EnsembleDistil cache:',sp_cache)
            if splade_ds is None:
                print('[ArguAna] generating naver/splade-cocondenser-ensembledistil top-500')
                splade_ds=generate_splade_generic({q:qmap_ds[q] for q in qids_ds},dmap_ds,top_k=500)
                sp_cache.write_text(json.dumps(splade_ds),encoding='utf-8')

    qids_ds=sorted(set(qids_ds)&set(dense_ds)&set(bm25_family_ds['bm25']))
    assert len(qids_ds)==EXPECTED_Q[ds_name], (ds_name,len(qids_ds))
    assert all(len(dense_ds[q])==min(DENSE_TOPK_DEEP,len(dmap_ds)) for q in qids_ds), f'{ds_name}: deep dense depth mismatch'
    DS_CANON[ds_name]={
        'qids':qids_ds,'qmap':qmap_ds,'dmap':dmap_ds,'qrels':qrels_ds,
        'dense':dense_ds,'bm25_family':bm25_family_ds,'splade':splade_ds,
        'runs':{'bm25':bm25_family_ds['bm25'],'dense':dense_ds},
    }
    if splade_ds is not None: DS_CANON[ds_name]['runs']['splade']=splade_ds

preflight=[]
for ds,d in DS_CANON.items():
    preflight.append({
        'dataset':ds,'queries':len(d['qids']),'documents':len(d['dmap']),
        'dense_generated_depth':max(map(len,d['dense'].values())),
        'dense_primary_cap':STABLE_DENSE_CAP_PRIMARY,'splade':d['splade'] is not None,
        'model_sha256':_locked['model_sha256'],
    })
preflight_df=pd.DataFrame(preflight)
preflight_df.to_csv(CANON_TABLES/'canonical_input_preflight.csv',index=False)
display(preflight_df)

# Release GPU model before the fusion-only analyses.
del STRICT_ENC
if torch.cuda.is_available(): torch.cuda.empty_cache()


## 18. Exact replication and real BM25-family experiment

The previous canonical run stopped here because its exact-copy assertion compared exact-collapse and MC-RRF against **ordinary RRF's** pre-copy ranking. That is not the invariance being claimed. This corrected cell compares every method with its own pre-copy output, matching the already successful SciFact diagnostic, and then runs the real BM25-family 8-test analysis.

In [ ]:
# Exact-copy diagnostic. IMPORTANT: every method is compared with its OWN pre-copy baseline.
# `copies` is total represented copies of the attacked list; copies=2 means one added exact copy.
exact_rows=[]
for ds_name,data in DS_CANON.items():
    sources=['bm25']+(['splade'] if data['splade'] is not None else [])+['dense']
    for attacked in sources:
        for L in [20,50,100]:
            for copies in [1,2,4,8]:
                ordinary_keep=[]; ordinary_set_keep=[]; exact_keep=[]; mc_keep=[]
                for q in data['qids']:
                    rankings={s:data['runs'][s][q] for s in sources}
                    depths={s:min(L,len(rankings[s])) for s in sources}

                    # Pre-copy reference for EACH method.
                    base_ord=[d for d,_ in rrf_from_prefixes(rankings,depths,k=60)[:10]]
                    base_exact=[d for d,_ in invariant_rrf_exact(rankings,depths,k=60)[0][:10]]
                    base_fm=provenance_family_map(rankings)
                    base_mc=[d for d,_ in mc_rrf(rankings,depths,k=60,family_map=base_fm)[0][:10]]

                    dup=dict(rankings); dd=dict(depths); fm=dict(base_fm)
                    for ci in range(1,copies):
                        name=f'{attacked}__copy{ci}'
                        dup[name]=rankings[attacked]
                        dd[name]=depths[attacked]
                        fm[name]=canonical_family_name(attacked)

                    ordinary=[d for d,_ in rrf_from_prefixes(dup,dd,k=60)[:10]]
                    exact=[d for d,_ in invariant_rrf_exact(dup,dd,k=60)[0][:10]]
                    mc=[d for d,_ in mc_rrf(dup,dd,k=60,family_map=fm)[0][:10]]

                    ordinary_keep.append(int(ordinary==base_ord))
                    ordinary_set_keep.append(int(set(ordinary)==set(base_ord)))
                    exact_keep.append(int(exact==base_exact))
                    mc_keep.append(int(mc==base_mc))

                row={
                    'dataset':ds_name,'attacked_source':attacked,'depth':L,'copies':copies,
                    'ordinary_order_preservation':float(np.mean(ordinary_keep)),
                    'ordinary_set_preservation':float(np.mean(ordinary_set_keep)),
                    'exact_collapse_order_preservation':float(np.mean(exact_keep)),
                    'mc_order_preservation':float(np.mean(mc_keep)),
                }
                exact_rows.append(row)

exact_df=pd.DataFrame(exact_rows)

# This is an algorithmic invariant. If it fails now, expose offending rows before stopping.
bad_exact=exact_df[(exact_df.exact_collapse_order_preservation<1.0-1e-15)|(exact_df.mc_order_preservation<1.0-1e-15)]
if len(bad_exact):
    display(bad_exact)
    raise AssertionError('Exact-collapse or MC-RRF failed exact-copy self-invariance. See rows above.')

exact_df.to_csv(CANON_TABLES/'canonical_exact_replication.csv',index=False)
paper_exact=exact_df[(exact_df.depth==50)&(exact_df.copies==2)].copy()
paper_exact.to_csv(CANON_TABLES/'canonical_exact_copy_depth50_one_added_copy.csv',index=False)
exact_dataset=paper_exact.groupby('dataset',as_index=False).agg(
    ordinary_order_preservation=('ordinary_order_preservation','mean'),
    ordinary_set_preservation=('ordinary_set_preservation','mean'),
    exact_collapse_order_preservation=('exact_collapse_order_preservation','mean'),
    mc_order_preservation=('mc_order_preservation','mean'),
)
exact_dataset['ordinary_order_change']=1-exact_dataset.ordinary_order_preservation
exact_dataset.to_csv(CANON_TABLES/'canonical_exact_copy_dataset_summary.csv',index=False)
display(paper_exact)
display(exact_dataset)

# Real BM25 family.
family_rows=[]; family_perq=[]; family_tests=[]
for ds_name in ['SciFact','FiQA','ArguAna']:
    data=DS_CANON[ds_name]; L=50
    base_sources=['bm25']+(['splade'] if data['splade'] is not None else [])+['dense']
    expanded_sources=['bm25','bm25_lowb','bm25_highb']+(['splade'] if data['splade'] is not None else [])+['dense']
    ord_set=[];mc_set=[];ord_order=[];mc_order=[];ord_delta=[];mc_delta=[];mc_vs=[];rbos=[]
    for q in data['qids']:
        br={s:data['runs'][s][q] if s in data['runs'] else data['bm25_family'][s][q] for s in base_sources}
        ar={s:(data['bm25_family'][s][q] if s.startswith('bm25') else data['runs'][s][q]) for s in expanded_sources}
        bd={s:min(L,len(v)) for s,v in br.items()}; ad={s:min(L,len(v)) for s,v in ar.items()}
        clean=rrf_from_prefixes(br,bd,k=60); ordinary=rrf_from_prefixes(ar,ad,k=60)
        fm={s:('BM25-family' if s.startswith('bm25') else s) for s in ar}
        mc,_=mc_rrf(ar,ad,k=60,family_map=fm)
        ct=[d for d,_ in clean[:10]]; ot=[d for d,_ in ordinary[:10]]; mt=[d for d,_ in mc[:10]]
        cn=ndcg_at_k(ct,data['qrels'][q],10); on=ndcg_at_k(ot,data['qrels'][q],10); mn=ndcg_at_k(mt,data['qrels'][q],10)
        ord_set.append(int(set(ot)==set(ct)));mc_set.append(int(set(mt)==set(ct)));ord_order.append(int(ot==ct));mc_order.append(int(mt==ct))
        ord_delta.append(on-cn);mc_delta.append(mn-cn);mc_vs.append(mn-on)
        b0=prefix_docs(data['bm25_family']['bm25'][q],L);b1=prefix_docs(data['bm25_family']['bm25_lowb'][q],L);b2=prefix_docs(data['bm25_family']['bm25_highb'][q],L)
        rbos.append(np.mean([rbo_finite(b0,b1,p=.9,depth=L),rbo_finite(b0,b2,p=.9,depth=L),rbo_finite(b1,b2,p=.9,depth=L)]))
        family_perq.append({'dataset':ds_name,'qid':q,'ordinary_delta':on-cn,'mc_delta':mn-cn,'mc_vs_ordinary':mn-on})
    family_rows.append({'dataset':ds_name,'depth':L,'mean_rbo':np.mean(rbos),'ordinary_set':np.mean(ord_set),'mc_set':np.mean(mc_set),'ordinary_order':np.mean(ord_order),'mc_order':np.mean(mc_order),'ordinary_delta':np.mean(ord_delta),'mc_delta':np.mean(mc_delta)})
    for test_name,arr,off in [('ordinary_family_effect',np.asarray(ord_delta),0),('MC_vs_ordinary_attacked',np.asarray(mc_vs),10000)]:
        mean,lo,hi=bootstrap_mean_ci(arr,seed=STAT_SEED+off+sum(map(ord,ds_name)));_,p=wilcoxon_safe(arr)
        family_tests.append({'dataset':ds_name,'test':test_name,'mean_difference':mean,'ci_low':lo,'ci_high':hi,'p_raw':p,'rank_biserial':paired_rank_biserial(arr)})
    if data['splade'] is not None:
        arr=[]
        for q in data['qids']:
            before={'bm25':data['runs']['bm25'][q],'dense':data['runs']['dense'][q]}; after={**before,'splade':data['runs']['splade'][q]}
            bd={s:min(50,len(v)) for s,v in before.items()}; ad={s:min(50,len(v)) for s,v in after.items()}
            bt=[d for d,_ in rrf_from_prefixes(before,bd,k=60)[:10]];at=[d for d,_ in rrf_from_prefixes(after,ad,k=60)[:10]]
            arr.append(ndcg_at_k(at,data['qrels'][q],10)-ndcg_at_k(bt,data['qrels'][q],10))
        arr=np.asarray(arr);mean,lo,hi=bootstrap_mean_ci(arr,seed=STAT_SEED+20000+sum(map(ord,ds_name)));_,p=wilcoxon_safe(arr)
        family_tests.append({'dataset':ds_name,'test':'SPLADE_new_family_effect','mean_difference':mean,'ci_low':lo,'ci_high':hi,'p_raw':p,'rank_biserial':paired_rank_biserial(arr)})

family_df=pd.DataFrame(family_rows); family_perq_df=pd.DataFrame(family_perq); family_test_df=pd.DataFrame(family_tests)
assert len(family_test_df)==8, family_test_df[['dataset','test']]
family_test_df['p_holm']=holm_adjust(family_test_df.p_raw.to_numpy(float));family_test_df['significant_0.05']=family_test_df.p_holm<.05
family_df.to_csv(CANON_TABLES/'canonical_real_bm25_family.csv',index=False)
family_perq_df.to_csv(CANON_TABLES/'canonical_real_bm25_family_per_query.csv',index=False)
family_test_df.to_csv(CANON_STATS/'canonical_real_family_holm8.csv',index=False)
display(family_df);display(family_test_df)


## 19. Canonical source-count scaling and primary 8-test inference

This reruns family sizes 1, 2, 4, and 8 for attacks on BM25 and dense evidence under both the original mild 5% and moderate 15% adjacent-swap conditions. The primary inferential endpoint remains the query-level difference in absolute nDCG drift at **mild, family size 8**.

In [ ]:
SCALING_PERTURBATIONS={'mild':0.05,'moderate':0.15}
scaling_rows=[]; primary_perq=[]; primary_tests=[]
for ds_name,data in DS_CANON.items():
    for attacked in ['bm25','dense']:
        other='dense' if attacked=='bm25' else 'bm25'
        for perturbation,rate in SCALING_PERTURBATIONS.items():
            for m in [1,2,4,8]:
                os=[];ms=[];oo=[];mo=[];osig=[];msig=[];oabs=[];mabs=[];rbos=[]
                for q in data['qids']:
                    ba=data['runs'][attacked][q];bo=data['runs'][other][q]
                    clean=rrf_from_rankings({attacked:ba,other:bo},100,60);ct=[d for d,_ in clean[:10]];cn=ndcg_at_k(ct,data['qrels'][q],10)
                    # Exact original controlled-scaling seed structure: dataset, qid, attacked source, perturbation, variant index.
                    fam=make_redundant_family(ba,m,rate,(ds_name,q,attacked,perturbation))
                    expanded={f'{attacked}__{n}':s for n,s in fam.items()};expanded[other]=bo
                    fnames=[n for n in expanded if n.startswith(attacked+'__')]
                    ordinary=rrf_from_rankings(expanded,100,60);mc=rrf_from_rankings(expanded,100,60,mc_family_weights(fnames,other))
                    ot=[d for d,_ in ordinary[:10]];mt=[d for d,_ in mc[:10]];on=ndcg_at_k(ot,data['qrels'][q],10);mn=ndcg_at_k(mt,data['qrels'][q],10)
                    os.append(int(set(ot)==set(ct)));ms.append(int(set(mt)==set(ct)));oo.append(int(ot==ct));mo.append(int(mt==ct))
                    osig.append(on-cn);msig.append(mn-cn);oabs.append(abs(on-cn));mabs.append(abs(mn-cn))
                    if m>1:
                        parent=prefix_docs(ba,100)
                        for nm,seq in fam.items():
                            if nm!='original':rbos.append(rbo_finite(parent,prefix_docs(seq,100),p=.9,depth=100))
                    if perturbation=='mild' and m==8:
                        primary_perq.append({
                            'dataset':ds_name,'attacked_source':attacked,'qid':q,
                            'ordinary_abs_drift':abs(on-cn),'mc_abs_drift':abs(mn-cn),
                            'ordinary_signed_delta':on-cn,'mc_signed_delta':mn-cn,
                            'ordinary_set_preserved':int(set(ot)==set(ct)),
                            'mc_set_preserved':int(set(mt)==set(ct)),
                        })
                scaling_rows.append({
                    'dataset':ds_name,'attacked_source':attacked,'perturbation':perturbation,'rate':rate,'family_size':m,
                    'mean_family_rbo':np.mean(rbos) if rbos else 1.0,
                    'ordinary_set_preservation':np.mean(os),'mc_set_preservation':np.mean(ms),
                    'ordinary_order_preservation':np.mean(oo),'mc_order_preservation':np.mean(mo),
                    'ordinary_signed_ndcg_delta':np.mean(osig),'mc_signed_ndcg_delta':np.mean(msig),
                    'ordinary_abs_ndcg_drift':np.mean(oabs),'mc_abs_ndcg_drift':np.mean(mabs),
                    'ordinary_family_mass':m/(m+1.0),'mc_family_mass':0.5,
                })

scaling_df=pd.DataFrame(scaling_rows); perq_df=pd.DataFrame(primary_perq)
assert len(perq_df)==sum(len(d['qids']) for d in DS_CANON.values())*2, 'Primary per-query scaling rows incomplete.'
for (ds,src),g in perq_df.groupby(['dataset','attacked_source'],sort=True):
    diff=(g.mc_abs_drift-g.ordinary_abs_drift).to_numpy(float)
    mean,lo,hi=bootstrap_mean_ci(diff,seed=STAT_SEED+sum(map(ord,ds+src)));_,p=wilcoxon_safe(diff)
    primary_tests.append({'dataset':ds,'attacked_source':src,'mean_mc_minus_ordinary_abs_drift':mean,'ci_low':lo,'ci_high':hi,'p_raw':p,'rank_biserial':paired_rank_biserial(diff)})
primary_test_df=pd.DataFrame(primary_tests)
assert len(primary_test_df)==8
primary_test_df['p_holm']=holm_adjust(primary_test_df.p_raw.to_numpy(float));primary_test_df['significant_0.05']=primary_test_df.p_holm<.05
primary_df2=scaling_df[(scaling_df.perturbation=='mild')&(scaling_df.family_size==8)].merge(primary_test_df,on=['dataset','attacked_source'])
primary_df2['drift_reduction']=primary_df2.ordinary_abs_ndcg_drift-primary_df2.mc_abs_ndcg_drift
primary_df2['set_preservation_gain']=primary_df2.mc_set_preservation-primary_df2.ordinary_set_preservation

scaling_macro=scaling_df.groupby(['attacked_source','perturbation','family_size'],as_index=False).agg(
    mean_rbo=('mean_family_rbo','mean'),
    ordinary_set_preservation=('ordinary_set_preservation','mean'),
    mc_set_preservation=('mc_set_preservation','mean'),
    ordinary_abs_ndcg_drift=('ordinary_abs_ndcg_drift','mean'),
    mc_abs_ndcg_drift=('mc_abs_ndcg_drift','mean'),
    ordinary_family_mass=('ordinary_family_mass','mean'),
    mc_family_mass=('mc_family_mass','mean'),
)

scaling_df.to_csv(CANON_TABLES/'canonical_scaling_all_sizes.csv',index=False)
scaling_macro.to_csv(CANON_TABLES/'canonical_scaling_macro_summary.csv',index=False)
perq_df.to_csv(CANON_TABLES/'canonical_scaling_primary_per_query.csv',index=False)
primary_df2.to_csv(CANON_TABLES/'canonical_scaling_primary.csv',index=False)
primary_test_df.to_csv(CANON_STATS/'canonical_scaling_holm8.csv',index=False)

display(primary_df2)
display(scaling_macro)
print('Primary lower drift:',int((primary_df2.drift_reduction>0).sum()),'/8')
print('Primary higher set:',int((primary_df2.set_preservation_gain>0).sum()),'/8')
print('Primary Holm significant:',int(primary_df2['significant_0.05'].sum()),'/8')
print('Macro primary drift reduction:',primary_df2.drift_reduction.mean())
print('Macro primary set gain:',primary_df2.set_preservation_gain.mean())


## 20. Original perturbation, `k x K`, and provenance protocols under the strict dense checkpoint

These cells intentionally restore the exact protocol keys from the limitation-management notebook. In particular, `k x K` uses the dedicated `kK` seed key and computes nDCG at the requested output depth K.


In [ ]:
# Exact perturbation functions from the limitation-management notebook.
def perturb_adjacent_canon(seq,rate,seed,depth=100):
    docs=prefix_docs(seq,depth);n=len(docs)
    if n<2 or rate<=0:return [(d,0.0) for d in docs]
    rng=np.random.default_rng(seed);n_swaps=max(1,int(round(rate*n)));positions=np.arange(n-1)
    chosen=sorted(rng.choice(positions,size=min(n_swaps,len(positions)),replace=False).tolist());out=list(docs);last=-2
    for p in chosen:
        if p==last+1:continue
        out[p],out[p+1]=out[p+1],out[p];last=p
    if out==docs:out[0],out[1]=out[1],out[0]
    assert set(out)==set(docs);return [(d,0.0) for d in out]

def perturb_bounded_canon(seq,rate,seed,depth=100,max_disp=5):
    docs=prefix_docs(seq,depth);n=len(docs)
    if n<2 or rate<=0:return [(d,0.0) for d in docs]
    rng=np.random.default_rng(seed);out=list(docs);n_moves=max(1,int(round(rate*n)));selected=rng.choice(np.arange(n),size=min(n_moves,n),replace=False)
    for original_pos in selected:
        cur=out.index(docs[int(original_pos)]);disp=0
        while disp==0:disp=int(rng.integers(-max_disp,max_disp+1))
        new=max(0,min(n-1,cur+disp));item=out.pop(cur);out.insert(new,item)
    if out==docs:out[0],out[1]=out[1],out[0]
    assert set(out)==set(docs);return [(d,0.0) for d in out]

def perturb_block_canon(seq,rate,seed,depth=100,block_size=5):
    docs=prefix_docs(seq,depth);n=len(docs)
    if n<2 or rate<=0:return [(d,0.0) for d in docs]
    rng=np.random.default_rng(seed);out=list(docs);blocks=[(s,min(n,s+block_size)) for s in range(0,n,block_size) if min(n,s+block_size)-s>=2]
    n_blocks=max(1,int(round(rate*n/max(2,block_size))));chosen=rng.choice(np.arange(len(blocks)),size=min(n_blocks,len(blocks)),replace=False)
    for bi in chosen:
        s,e=blocks[int(bi)];block=out[s:e];perm=list(rng.permutation(block))
        if perm==block:perm=block[1:]+block[:1]
        out[s:e]=perm
    if out==docs:out[0],out[1]=out[1],out[0]
    assert set(out)==set(docs);return [(d,0.0) for d in out]

PERT_CANON={'adjacent_swap':perturb_adjacent_canon,'bounded_displacement':perturb_bounded_canon,'block_permutation':perturb_block_canon}
def make_family_canon(base_seq,size,rate,mechanism,key):
    fam={'original':base_seq};fn=PERT_CANON[mechanism]
    for j in range(1,int(size)):fam[f'variant{j}']=fn(base_seq,rate,deterministic_seed(*key,mechanism,j),depth=100)
    return fam

# A. Perturbation mechanism sensitivity: mild inference + moderate descriptive.
pert_rows=[];pert_perq=[]
for ds_name,data in DS_CANON.items():
    for attacked in ['bm25','dense']:
        other='dense' if attacked=='bm25' else 'bm25'
        for mech in PERT_CANON:
            for level,rate in {'mild':.05,'moderate':.15}.items():
                os=[];ms=[];oa=[];ma=[];rbos=[]
                for q in data['qids']:
                    ba=data['runs'][attacked][q];bo=data['runs'][other][q];clean=rrf_from_rankings({attacked:ba,other:bo},100,60);ct=[d for d,_ in clean[:10]];cn=ndcg_at_k(ct,data['qrels'][q],10)
                    fam=make_family_canon(ba,8,rate,mech,(ds_name,q,attacked,level));ex={f'{attacked}__{n}':s for n,s in fam.items()};ex[other]=bo;names=[n for n in ex if n.startswith(attacked+'__')]
                    o=rrf_from_rankings(ex,100,60);m=rrf_from_rankings(ex,100,60,mc_family_weights(names,other));ot=[d for d,_ in o[:10]];mt=[d for d,_ in m[:10]]
                    on=ndcg_at_k(ot,data['qrels'][q],10);mn=ndcg_at_k(mt,data['qrels'][q],10);os.append(int(set(ot)==set(ct)));ms.append(int(set(mt)==set(ct)));oa.append(abs(on-cn));ma.append(abs(mn-cn))
                    parent=prefix_docs(ba,100)
                    for nm,seq in fam.items():
                        if nm!='original':rbos.append(rbo_finite(parent,prefix_docs(seq,100),p=.9,depth=100))
                    if level=='mild':pert_perq.append({'dataset':ds_name,'attacked_source':attacked,'mechanism':mech,'qid':q,'ordinary_abs_drift':abs(on-cn),'mc_abs_drift':abs(mn-cn),'ordinary_set_preserved':int(set(ot)==set(ct)),'mc_set_preserved':int(set(mt)==set(ct))})
                pert_rows.append({'dataset':ds_name,'attacked_source':attacked,'mechanism':mech,'perturbation':level,'mean_rbo':np.mean(rbos),'ordinary_set':np.mean(os),'mc_set':np.mean(ms),'ordinary_abs_drift':np.mean(oa),'mc_abs_drift':np.mean(ma)})
pert_df=pd.DataFrame(pert_rows);pert_perq_df=pd.DataFrame(pert_perq);stats=[]
for (ds,src,mech),g in pert_perq_df.groupby(['dataset','attacked_source','mechanism'],sort=True):
    diff=(g.mc_abs_drift-g.ordinary_abs_drift).to_numpy(float);mean,lo,hi=bootstrap_mean_ci(diff,seed=STAT_SEED+sum(map(ord,ds+src+mech)));_,p=wilcoxon_safe(diff)
    stats.append({'dataset':ds,'attacked_source':src,'mechanism':mech,'mean_mc_minus_ordinary_abs_drift':mean,'ci_low':lo,'ci_high':hi,'p_raw':p,'rank_biserial':paired_rank_biserial(diff),'ordinary_set':g.ordinary_set_preserved.mean(),'mc_set':g.mc_set_preserved.mean(),'ordinary_abs_drift':g.ordinary_abs_drift.mean(),'mc_abs_drift':g.mc_abs_drift.mean()})
pert_stats=pd.DataFrame(stats);pert_stats['p_holm_24']=holm_adjust(pert_stats.p_raw.to_numpy(float));pert_stats['significant_0.05']=pert_stats.p_holm_24<.05;pert_stats['drift_reduction']=pert_stats.ordinary_abs_drift-pert_stats.mc_abs_drift;pert_stats['set_gain']=pert_stats.mc_set-pert_stats.ordinary_set
pert_macro=pert_df[pert_df.perturbation=='mild'].groupby('mechanism',as_index=False).agg(mean_rbo=('mean_rbo','mean'),ordinary_set=('ordinary_set','mean'),mc_set=('mc_set','mean'),ordinary_abs_drift=('ordinary_abs_drift','mean'),mc_abs_drift=('mc_abs_drift','mean'));pert_macro['set_gain']=pert_macro.mc_set-pert_macro.ordinary_set;pert_macro['drift_reduction']=pert_macro.ordinary_abs_drift-pert_macro.mc_abs_drift
pert_df.to_csv(CANON_TABLES/'canonical_perturbation_summary.csv',index=False);pert_stats.to_csv(CANON_STATS/'canonical_perturbation_holm24.csv',index=False);pert_macro.to_csv(CANON_TABLES/'canonical_perturbation_macro.csv',index=False)
print('Perturbation: lower drift',int((pert_stats.drift_reduction>0).sum()),'/24; higher set',int((pert_stats.set_gain>0).sum()),'/24; Holm significant',int(pert_stats['significant_0.05'].sum()),'/24');display(pert_macro)

# B. Original k x K protocol.
kk_cond=[];kk_grid=[]
for kval in [10,30,60,100]:
    for Kout in [5,10,20]:
        for ds_name,data in DS_CANON.items():
            for attacked in ['bm25','dense']:
                other='dense' if attacked=='bm25' else 'bm25';oa=[];ma=[];os=[];ms=[]
                for q in data['qids']:
                    ba=data['runs'][attacked][q];bo=data['runs'][other][q];clean=rrf_from_rankings({attacked:ba,other:bo},100,kval);ct=[d for d,_ in clean[:Kout]];cn=ndcg_at_k(ct,data['qrels'][q],Kout)
                    fam=make_family_canon(ba,8,.05,'adjacent_swap',(ds_name,q,attacked,'kK'));ex={f'{attacked}__{n}':s for n,s in fam.items()};ex[other]=bo;names=[n for n in ex if n.startswith(attacked+'__')]
                    o=rrf_from_rankings(ex,100,kval);m=rrf_from_rankings(ex,100,kval,mc_family_weights(names,other));ot=[d for d,_ in o[:Kout]];mt=[d for d,_ in m[:Kout]]
                    oa.append(abs(ndcg_at_k(ot,data['qrels'][q],Kout)-cn));ma.append(abs(ndcg_at_k(mt,data['qrels'][q],Kout)-cn));os.append(int(set(ot)==set(ct)));ms.append(int(set(mt)==set(ct)))
                kk_cond.append({'dataset':ds_name,'attacked_source':attacked,'k':kval,'K':Kout,'ordinary_abs_drift':np.mean(oa),'mc_abs_drift':np.mean(ma),'ordinary_set':np.mean(os),'mc_set':np.mean(ms)})
kk_cond_df=pd.DataFrame(kk_cond)
for (kval,Kout),g in kk_cond_df.groupby(['k','K'],sort=True):
    kk_grid.append({'k':kval,'K':Kout,'n_conditions':len(g),'conditions_mc_lower_abs_drift':int((g.mc_abs_drift<g.ordinary_abs_drift).sum()),'conditions_mc_higher_set_preservation':int((g.mc_set>g.ordinary_set).sum()),'macro_set_gain':float((g.mc_set-g.ordinary_set).mean()),'macro_abs_drift_reduction':float((g.ordinary_abs_drift-g.mc_abs_drift).mean())})
kk_grid_df=pd.DataFrame(kk_grid);kk_cond_df.to_csv(CANON_TABLES/'canonical_k_by_K_conditions.csv',index=False);kk_grid_df.to_csv(CANON_TABLES/'canonical_k_by_K_sensitivity.csv',index=False);display(kk_grid_df)

# C. Provenance under-grouping and over-grouping.
def weights_from_groups(group_map):
    groups=defaultdict(list)
    for s,g in group_map.items():groups[str(g)].append(s)
    w={}
    for _,members in groups.items():
        for s in members:w[s]=1.0/len(members)
    return w

def split_group_map(fam_names,other,split_rate):
    fam_names=sorted(fam_names);original=[n for n in fam_names if n.endswith('__original')];variants=[n for n in fam_names if n not in original];n_split=int(round(float(split_rate)*len(variants)));split=set(variants[:n_split]);g={other:'OTHER'}
    for n in fam_names:g[n]=f'SPLIT::{n}' if n in split else 'ATTACKED_CORE'
    return g

prov=[];over=[]
for ds_name,data in DS_CANON.items():
    for attacked in ['bm25','dense']:
        other='dense' if attacked=='bm25' else 'bm25'
        for rate in [0,.25,.5,.75,1.0]:
            sp=[];dr=[];mass=[]
            for q in data['qids']:
                ba=data['runs'][attacked][q];bo=data['runs'][other][q];clean=rrf_from_rankings({attacked:ba,other:bo},100,60);ct=[d for d,_ in clean[:10]];cn=ndcg_at_k(ct,data['qrels'][q],10)
                fam=make_family_canon(ba,8,.05,'adjacent_swap',(ds_name,q,attacked,'prov'));ex={f'{attacked}__{n}':s for n,s in fam.items()};ex[other]=bo;fn=[n for n in ex if n.startswith(attacked+'__')];w=weights_from_groups(split_group_map(fn,other,rate));f=rrf_from_rankings(ex,100,60,w);top=[d for d,_ in f[:10]]
                sp.append(int(set(top)==set(ct)));dr.append(abs(ndcg_at_k(top,data['qrels'][q],10)-cn));mass.append(sum(w[n] for n in fn)/sum(w.values()))
            prov.append({'dataset':ds_name,'attacked_source':attacked,'split_rate':rate,'attacked_mass':np.mean(mass),'set_preservation':np.mean(sp),'abs_drift':np.mean(dr)})
        ca=[];oa2=[];cs=[];os2=[]
        for q in data['qids']:
            ba=data['runs'][attacked][q];bo=data['runs'][other][q];clean=rrf_from_rankings({attacked:ba,other:bo},100,60);ct=[d for d,_ in clean[:10]];cn=ndcg_at_k(ct,data['qrels'][q],10)
            fam=make_family_canon(ba,8,.05,'adjacent_swap',(ds_name,q,attacked,'over'));ex={f'{attacked}__{n}':s for n,s in fam.items()};ex[other]=bo;fn=[n for n in ex if n.startswith(attacked+'__')]
            correct_map={n:'ATTACKED' for n in fn};correct_map[other]='OTHER';over_map={n:'MERGED_ALL' for n in ex}
            c=rrf_from_rankings(ex,100,60,weights_from_groups(correct_map));o=rrf_from_rankings(ex,100,60,weights_from_groups(over_map));ct2=[d for d,_ in c[:10]];ot2=[d for d,_ in o[:10]]
            ca.append(abs(ndcg_at_k(ct2,data['qrels'][q],10)-cn));oa2.append(abs(ndcg_at_k(ot2,data['qrels'][q],10)-cn));cs.append(int(set(ct2)==set(ct)));os2.append(int(set(ot2)==set(ct)))
        over.append({'dataset':ds_name,'attacked_source':attacked,'correct_set_preservation':np.mean(cs),'overgroup_set_preservation':np.mean(os2),'correct_abs_drift':np.mean(ca),'overgroup_abs_drift':np.mean(oa2),'overgroup_minus_correct_abs_drift':np.mean(oa2)-np.mean(ca)})
prov_df2=pd.DataFrame(prov);over_df=pd.DataFrame(over);prov_macro2=prov_df2.groupby('split_rate',as_index=False).agg(attacked_mass=('attacked_mass','mean'),set_preservation=('set_preservation','mean'),abs_drift=('abs_drift','mean'))
prov_df2.to_csv(CANON_TABLES/'canonical_provenance_splitting.csv',index=False);prov_macro2.to_csv(CANON_TABLES/'canonical_provenance_macro.csv',index=False);over_df.to_csv(CANON_TABLES/'canonical_provenance_overgrouping.csv',index=False);display(prov_macro2);display(over_df)


## 21. StableRRF under the canonical strict dense runs

This repeats the exact certificate with the same observation caps. The scheduler remains secondary. The cell now regenerates both the all-query policy summaries and the original **16 paired adaptive-versus-alternative policy tests with Holm correction**, plus the two-source finite-lattice oracle audit.

In [ ]:
# Primary adaptive set/order results.
stable=[];static=[]
for ds_name,data in DS_CANON.items():
    source_sets=[('BM25+Dense',['bm25','dense'])]
    if data['splade'] is not None:source_sets.append(('BM25+SPLADE+Dense',['bm25','splade','dense']))
    for label,sources in source_sets:
        qids=sorted(set(data['qids']).intersection(*(set(data['runs'][s]) for s in sources)));caps={'bm25':1000,'dense':STABLE_DENSE_CAP_PRIMARY,'splade':500}
        for target in ['order','set']:
            cert=[];reads=[];budgets=[]
            for q in qids:
                rankings={s:data['runs'][s][q] for s in sources};maxd={s:min(caps[s],len(rankings[s])) for s in sources}
                res=stable_rrf_adaptive(rankings,K=10,k=60,target=target,start_depth=10,step=10,max_depths=maxd,n_outsiders=25)
                cert.append(int(res['certified']));reads.append(res['candidate_reads']);budgets.append(sum(maxd.values()))
            stable.append({
                'dataset':ds_name,'source_set':label,'target':target,'n_queries':len(qids),
                'cert_rate':np.mean(cert),'mean_candidate_reads':np.mean(reads),
                'mean_budget':np.mean(budgets),'saving_pct':100*np.mean(1-np.asarray(reads)/np.asarray(budgets)),
            })
    for Kout in [1,10]:
        vals=[]
        for q in data['qids']:
            rankings={'bm25':data['runs']['bm25'][q],'dense':data['runs']['dense'][q]};depths={s:min(50,len(v)) for s,v in rankings.items()}
            b,u,_=censored_rrf_bounds(rankings,depths,k=60);c=certify_topk(b,u,K=Kout);vals.append(int(c['certified_prefix_len']>=Kout))
        static.append({'dataset':ds_name,'K':Kout,'depth':50,'ordered_cert_rate':np.mean(vals)})
stable_df2=pd.DataFrame(stable);static_df2=pd.DataFrame(static)
stable_df2.to_csv(CANON_TABLES/'canonical_stablerrf.csv',index=False);static_df2.to_csv(CANON_TABLES/'canonical_static_depth50.csv',index=False)
display(stable_df2);display(static_df2)

# Scheduler policy audit, ordered top-10 only. This reproduces the original 5-policy audit.
def stable_policy_run_canon(rankings,policy,qkey,K=10,k=60.0,start=10,step=10,max_depths=None,max_iter=10000):
    names=list(rankings);max_depths=max_depths or {n:len(rankings[n]) for n in names};depths={n:min(start,int(max_depths[n]),len(rankings[n])) for n in names};rr_idx=0;rng=np.random.default_rng(deterministic_seed('stable',qkey,policy))
    for _ in range(max_iter):
        b,u,meta=censored_rrf_bounds(rankings,depths,k=k);c=certify_topk(b,u,K=K);cost=observed_cost(rankings,depths)
        if c['certified_order']:return {'certified':True,**cost}
        expandable=[n for n in names if depths[n]<min(len(rankings[n]),int(max_depths[n]))]
        if not expandable:return {'certified':False,**cost}
        if policy=='adaptive':chosen,_=choose_ranker_to_deepen(rankings,depths,b,meta,K=K,step=step,max_depths=max_depths,n_outsiders=25)
        elif policy=='round_robin':
            ordered=sorted(names);chosen=None
            for _j in range(len(ordered)):
                cand=ordered[rr_idx%len(ordered)];rr_idx+=1
                if cand in expandable:chosen=cand;break
        elif policy=='bm25_first':chosen='bm25' if 'bm25' in expandable else sorted(expandable)[0]
        elif policy=='dense_first':chosen='dense' if 'dense' in expandable else sorted(expandable)[0]
        elif policy=='random':chosen=str(rng.choice(expandable))
        else:raise ValueError(policy)
        if chosen not in expandable:chosen=sorted(expandable,key=lambda n:(depths[n],n))[0]
        depths[chosen]=min(depths[chosen]+step,len(rankings[chosen]),int(max_depths[chosen]))
    raise RuntimeError('policy loop exceeded limit')

STABLE_POLICIES_CANON=['adaptive','round_robin','bm25_first','dense_first','random']
policy_perq=[]
for ds_name,data in DS_CANON.items():
    for q in data['qids']:
        rankings={'bm25':data['runs']['bm25'][q],'dense':data['runs']['dense'][q]}
        maxd={'bm25':min(1000,len(rankings['bm25'])),'dense':min(STABLE_DENSE_CAP_PRIMARY,len(rankings['dense']))}
        full_budget=sum(maxd.values())
        for policy in STABLE_POLICIES_CANON:
            r=stable_policy_run_canon(rankings,policy,(ds_name,q),max_depths=maxd)
            policy_perq.append({
                'dataset':ds_name,'qid':q,'policy':policy,'certified':r['certified'],
                'candidate_reads':r['candidate_reads'],'full_budget':full_budget,
                'saving':1-r['candidate_reads']/full_budget,
            })
policy_perq_df=pd.DataFrame(policy_perq)
policy_rows=[]
for (ds,policy),g in policy_perq_df.groupby(['dataset','policy'],sort=True):
    policy_rows.append({
        'dataset':ds,'policy':policy,'n_queries':len(g),
        'certification_rate':float(g.certified.mean()),
        'mean_candidate_reads':float(g.candidate_reads.mean()),
        'median_candidate_reads':float(g.candidate_reads.median()),
        'mean_saving':float(g.saving.mean()),
        'mean_reads_certified_only':float(g.loc[g.certified,'candidate_reads'].mean()) if g.certified.any() else np.nan,
    })
policy_df=pd.DataFrame(policy_rows)
policy_perq_df.to_csv(CANON_TABLES/'canonical_stable_policy_per_query.csv',index=False)
policy_df.to_csv(CANON_TABLES/'canonical_stable_policy_summary.csv',index=False)
display(policy_df)

# Original inferential family: adaptive minus each of the four alternatives, 4 datasets x 4 = 16 tests.
def paired_bootstrap_ci_canon(diffs,B=10000,seed=41,alpha=0.05):
    x=np.asarray(diffs,dtype=float)
    if len(x)==0:return (np.nan,np.nan)
    rng=np.random.default_rng(seed);n=len(x);means=np.empty(B,dtype=float)
    for i in range(B):means[i]=x[rng.integers(0,n,size=n)].mean()
    return tuple(np.quantile(means,[alpha/2,1-alpha/2]))

def rank_biserial_sign_canon(diffs):
    x=np.asarray(diffs,dtype=float);x=x[np.abs(x)>1e-15]
    if len(x)==0:return 0.0
    return float((np.sum(x>0)-np.sum(x<0))/len(x))

policy_stats=[]
for ds in sorted(policy_perq_df.dataset.unique()):
    a=policy_perq_df[(policy_perq_df.dataset==ds)&(policy_perq_df.policy=='adaptive')].set_index('qid')
    for policy in [p for p in STABLE_POLICIES_CANON if p!='adaptive']:
        b=policy_perq_df[(policy_perq_df.dataset==ds)&(policy_perq_df.policy==policy)].set_index('qid')
        common=a.index.intersection(b.index)
        diffs=(a.loc[common,'candidate_reads']-b.loc[common,'candidate_reads']).to_numpy(float)
        ci=paired_bootstrap_ci_canon(diffs,B=N_BOOT,seed=SEED)
        _,p=wilcoxon_safe(diffs)
        policy_stats.append({
            'dataset':ds,'comparison':f'adaptive_minus_{policy}','n_queries':len(common),
            'mean_read_difference':float(np.mean(diffs)),'ci_low':float(ci[0]),'ci_high':float(ci[1]),
            'p_raw':p,'rank_biserial_sign':rank_biserial_sign_canon(diffs),
        })
policy_stats_df=pd.DataFrame(policy_stats)
assert len(policy_stats_df)==16
policy_stats_df['p_holm_16']=holm_adjust(policy_stats_df.p_raw.to_numpy(float))
policy_stats_df['significant_holm_0.05']=policy_stats_df.p_holm_16<0.05
policy_stats_df.to_csv(CANON_STATS/'canonical_stable_policy_holm16.csv',index=False)
display(policy_stats_df)

# Finite-lattice oracle on a deterministic sample of at most 100 queries.
def depth_lattice(maxL,start=10,step=10):
    if maxL<=0:return [0]
    first=min(start,maxL);vals=list(range(first,maxL+1,step))
    if maxL not in vals:vals.append(maxL)
    return sorted(set(vals))

def grid_oracle_two(rankings,K=10,k=60,max_depths=None):
    names=list(rankings)
    max_depths=max_depths or {n:len(rankings[n]) for n in names}
    limits={n:min(len(rankings[n]),int(max_depths.get(n,len(rankings[n])))) for n in names}
    grids={n:depth_lattice(limits[n]) for n in names};cands=[]
    for combo in product(*(grids[n] for n in names)):
        depths={n:int(combo[i]) for i,n in enumerate(names)};cands.append((sum(depths.values()),tuple(depths[n] for n in names),depths))
    cands.sort(key=lambda x:(x[0],x[1]))
    for _,_,depths in cands:
        b,u,_=censored_rrf_bounds(rankings,depths,k=k);c=certify_topk(b,u,K=K)
        if c['certified_order']:return {'certified':True,'reads':sum(depths.values()),'depths':depths}
    return {'certified':False,'reads':sum(limits.values()),'depths':limits}

oracle_rows=[]
for ds_name,data in DS_CANON.items():
    rng=np.random.default_rng(deterministic_seed('oracle-sample',ds_name));qids=list(data['qids'])
    sample=qids if len(qids)<=100 else [qids[i] for i in sorted(rng.choice(np.arange(len(qids)),size=100,replace=False))]
    for q in sample:
        rankings={'bm25':data['runs']['bm25'][q],'dense':data['runs']['dense'][q]}
        adaptive=stable_rrf_adaptive(rankings,K=10,k=60,target='order',start_depth=10,step=10,max_depths={'bm25':min(1000,len(rankings['bm25'])),'dense':min(100,len(rankings['dense']))},n_outsiders=25)
        oracle=grid_oracle_two(rankings,K=10,k=60,max_depths={'bm25':min(1000,len(rankings['bm25'])),'dense':min(STABLE_DENSE_CAP_PRIMARY,len(rankings['dense']))})
        oracle_rows.append({'dataset':ds_name,'qid':q,'adaptive_certified':adaptive['certified'],'adaptive_reads':adaptive['candidate_reads'],'oracle_certified':oracle['certified'],'oracle_reads':oracle['reads']})
oracle_df=pd.DataFrame(oracle_rows)
oracle_df.to_csv(CANON_TABLES/'canonical_stable_two_source_oracle.csv',index=False)
oracle_summary=oracle_df.groupby('dataset',as_index=False).agg(
    n=('qid','count'),adaptive_cert_rate=('adaptive_certified','mean'),oracle_cert_rate=('oracle_certified','mean'),
    adaptive_reads=('adaptive_reads','mean'),oracle_reads=('oracle_reads','mean'))
oracle_summary['adaptive_minus_oracle_reads']=oracle_summary.adaptive_reads-oracle_summary.oracle_reads
oracle_summary.to_csv(CANON_TABLES/'canonical_stable_two_source_oracle_summary.csv',index=False)
display(oracle_summary)


## 22. Secondary StableRRF extension: deeper dense caps and exact-prefix diagnostics

The canonical StableRRF result remains `k=60` with the dense source capped at 100. This extension asks a separate operational question: **how much of the low certification rate is caused by the externally imposed dense observation cap?**

The same locked dense model is already ranked to 1000. We therefore repeat the exact two-source ordered certificate at dense caps 100, 250, 500, and 1000 while BM25 remains capped at 1000. No source is marked complete; unobserved tails retain their reciprocal upper bounds.

For efficiency, the notebook first evaluates the certificate at the maximum allowed depths. By monotone tightening, if the exact order certificate fails there, no scheduler can have certified earlier under the same caps, so the query is assigned the full budget without replaying the adaptive schedule. If the final certificate is feasible, the original adaptive scheduler is run to measure candidate reads.

We also retain the final exact ordered-prefix length. A query with prefix length 5 but no complete top-10 certificate has five mathematically fixed leading positions, not zero exact information.


In [ ]:

def final_order_state(rankings, max_depths, k=60.0, K=10):
    depths={n:min(int(max_depths[n]),len(rankings[n])) for n in rankings}
    b,u,meta=censored_rrf_bounds(rankings,depths,k=k)
    c=certify_topk(b,u,K=K)
    ordered_len=int(c['certified_prefix_len'])
    # Identify the first unresolved competitor type for diagnostics.
    blocker='none'
    first_margin=np.nan
    if ordered_len < K and len(b) > ordered_len:
        rows=sorted(b,key=lambda x:(-x.lb,-x.ub,x.docid))
        j=ordered_len
        row=rows[j]
        outsider_max=max([x.ub for x in rows[j+1:]]+[-np.inf])
        unseen=float(u)
        competitor=max(outsider_max,unseen)
        first_margin=float(row.lb-competitor)
        blocker='unseen_tail' if unseen>=outsider_max else 'observed_outsider'
    return {
        'order_feasible':bool(c['certified_order']),
        'set_feasible':bool(c['certified_set']),
        'certified_prefix_len':ordered_len,
        'unseen_ub':float(u),
        'blocker':blocker,
        'first_unresolved_margin':first_margin,
    }


def adaptive_reads_if_feasible(rankings,max_depths,k=60.0,K=10):
    state=final_order_state(rankings,max_depths,k=k,K=K)
    full_budget=sum(min(int(max_depths[n]),len(rankings[n])) for n in rankings)
    if not state['order_feasible']:
        return {**state,'candidate_reads':int(full_budget),'certified':False,'saving':0.0}
    res=stable_rrf_adaptive(
        rankings,K=K,k=k,target='order',start_depth=10,step=10,
        max_depths=max_depths,n_outsiders=25,
    )
    return {**state,'candidate_reads':int(res['candidate_reads']),'certified':bool(res['certified']),
            'saving':1.0-float(res['candidate_reads'])/float(full_budget)}


deep_cap_perq=[]
for ds_name,data in DS_CANON.items():
    for dense_cap in STABLE_DENSE_CAPS:
        for q in data['qids']:
            rankings={'bm25':data['runs']['bm25'][q],'dense':data['runs']['dense'][q]}
            maxd={'bm25':min(1000,len(rankings['bm25'])),'dense':min(int(dense_cap),len(rankings['dense']))}
            r=adaptive_reads_if_feasible(rankings,maxd,k=RRF_K,K=10)
            deep_cap_perq.append({
                'dataset':ds_name,'qid':q,'k':RRF_K,'dense_cap':int(dense_cap),
                'bm25_cap':int(maxd['bm25']),'actual_dense_cap':int(maxd['dense']),
                **r,
            })

deep_cap_perq_df=pd.DataFrame(deep_cap_perq)
deep_cap_summary=deep_cap_perq_df.groupby(['dataset','dense_cap'],as_index=False).agg(
    n_queries=('qid','count'),
    order_cert_rate=('order_feasible','mean'),
    set_cert_rate=('set_feasible','mean'),
    mean_candidate_reads=('candidate_reads','mean'),
    mean_saving=('saving','mean'),
    mean_final_prefix_len=('certified_prefix_len','mean'),
    median_final_prefix_len=('certified_prefix_len','median'),
    mean_unseen_ub=('unseen_ub','mean'),
)

prefix_threshold_rows=[]
for (ds,cap),g in deep_cap_perq_df.groupby(['dataset','dense_cap'],sort=True):
    row={'dataset':ds,'dense_cap':int(cap),'n_queries':len(g)}
    for j in [1,2,3,5,10]:
        row[f'prefix_at_least_{j}']=float((g.certified_prefix_len>=j).mean())
    row['observed_outsider_blocker_rate']=float((g.blocker=='observed_outsider').mean())
    row['unseen_tail_blocker_rate']=float((g.blocker=='unseen_tail').mean())
    prefix_threshold_rows.append(row)
prefix_summary_df=pd.DataFrame(prefix_threshold_rows)

deep_cap_perq_df.to_csv(CANON_TABLES/'secondary_stable_dense_cap_per_query.csv',index=False)
deep_cap_summary.to_csv(CANON_TABLES/'secondary_stable_dense_cap_summary.csv',index=False)
prefix_summary_df.to_csv(CANON_TABLES/'secondary_stable_prefix_summary.csv',index=False)

display(deep_cap_summary)
display(prefix_summary_df)
print('\nTREC-COVID dense-cap sensitivity:')
display(deep_cap_summary[deep_cap_summary.dataset=='TREC-COVID'])
display(prefix_summary_df[prefix_summary_df.dataset=='TREC-COVID'])


## 23. Secondary task-level adaptive `k` with ranking-only calibration

This experiment does **not** replace the paper's fixed `k=60` primary setting. It asks whether a deployment can configure one RRF constant per retrieval task without using relevance labels to choose it.

For each dataset, query IDs are deterministically split into 20% calibration and 80% held-out evaluation. The candidate grid is the previously used StableRRF sensitivity grid `{1,5,10,20,60,100,200}`. On calibration queries, each candidate `k` is scored using only the final exact-certificate feasibility at the canonical BM25=1000 / dense=100 caps. Selection is lexicographic:

1. highest exact ordered top-10 feasibility rate;
2. largest mean exact certified-prefix length;
3. closest to the primary `k=60` as a conservative tie-break.

No qrels or nDCG values enter selection. The selected task-level `k` is then frozen and evaluated on the held-out queries. Held-out qrels are used only afterward to report the deepest-observed RRF nDCG@10 difference from fixed `k=60`.

This is a **secondary task-calibration experiment**, not a claim that per-query adaptive `k` is part of StableRRF. Per-query changes to `k` would change the target fusion function itself and are deliberately excluded here.


In [ ]:

def task_k_split(ds_name,qids,fraction=TASK_K_CALIB_FRACTION):
    qids=list(map(str,qids))
    scored=sorted((deterministic_seed('task-k-calibration',SEED,ds_name,q),q) for q in qids)
    n=max(5,int(round(float(fraction)*len(scored))))
    n=min(n,max(1,len(scored)-1))
    calib=[q for _,q in scored[:n]]
    held=[q for _,q in scored[n:]]
    assert set(calib).isdisjoint(held) and set(calib)|set(held)==set(qids)
    return calib,held


def final_feasibility_for_k(data,qids,k,dense_cap=TASK_K_DENSE_CAP,K=10):
    rows=[]
    for q in qids:
        rankings={'bm25':data['runs']['bm25'][q],'dense':data['runs']['dense'][q]}
        maxd={'bm25':min(1000,len(rankings['bm25'])),'dense':min(int(dense_cap),len(rankings['dense']))}
        s=final_order_state(rankings,maxd,k=float(k),K=K)
        rows.append(s)
    return {
        'cert_rate':float(np.mean([x['order_feasible'] for x in rows])),
        'mean_prefix_len':float(np.mean([x['certified_prefix_len'] for x in rows])),
        'set_cert_rate':float(np.mean([x['set_feasible'] for x in rows])),
    }


def deepest_rrf_ndcg(data,q,k,dense_cap=TASK_K_DENSE_CAP,K=10):
    rankings={'bm25':data['runs']['bm25'][q],'dense':data['runs']['dense'][q]}
    depths={'bm25':min(1000,len(rankings['bm25'])),'dense':min(int(dense_cap),len(rankings['dense']))}
    top=[d for d,_ in rrf_from_prefixes(rankings,depths,k=float(k))[:K]]
    return float(ndcg_at_k(top,data['qrels'][q],K))


task_k_calibration=[];task_k_selection=[];task_k_eval_perq=[]
for ds_name,data in DS_CANON.items():
    calib_q,held_q=task_k_split(ds_name,data['qids'])
    candidates=[]
    for kval in TASK_K_GRID:
        met=final_feasibility_for_k(data,calib_q,kval,dense_cap=TASK_K_DENSE_CAP,K=10)
        row={'dataset':ds_name,'k':float(kval),'n_calibration':len(calib_q),**met}
        task_k_calibration.append(row);candidates.append(row)
    # Conservative ranking-only selection. No qrels are touched here.
    best=max(candidates,key=lambda r:(r['cert_rate'],r['mean_prefix_len'],-abs(float(r['k'])-float(RRF_K))))
    selected=float(best['k'])
    task_k_selection.append({
        'dataset':ds_name,'selected_k':selected,'n_calibration':len(calib_q),'n_heldout':len(held_q),
        'calibration_cert_rate':best['cert_rate'],'calibration_mean_prefix_len':best['mean_prefix_len'],
        'selection_used_qrels':False,
    })
    for q in held_q:
        rankings={'bm25':data['runs']['bm25'][q],'dense':data['runs']['dense'][q]}
        maxd={'bm25':min(1000,len(rankings['bm25'])),'dense':min(TASK_K_DENSE_CAP,len(rankings['dense']))}
        sel=adaptive_reads_if_feasible(rankings,maxd,k=selected,K=10)
        fixed=adaptive_reads_if_feasible(rankings,maxd,k=RRF_K,K=10)
        nd_sel=deepest_rrf_ndcg(data,q,selected,dense_cap=TASK_K_DENSE_CAP,K=10)
        nd_fix=deepest_rrf_ndcg(data,q,RRF_K,dense_cap=TASK_K_DENSE_CAP,K=10)
        task_k_eval_perq.append({
            'dataset':ds_name,'qid':q,'selected_k':selected,
            'selected_certified':sel['certified'],'fixed60_certified':fixed['certified'],
            'selected_prefix_len':sel['certified_prefix_len'],'fixed60_prefix_len':fixed['certified_prefix_len'],
            'selected_reads':sel['candidate_reads'],'fixed60_reads':fixed['candidate_reads'],
            'selected_saving':sel['saving'],'fixed60_saving':fixed['saving'],
            'selected_ndcg10':nd_sel,'fixed60_ndcg10':nd_fix,'ndcg10_delta_selected_minus_fixed60':nd_sel-nd_fix,
        })

task_k_calibration_df=pd.DataFrame(task_k_calibration)
task_k_selection_df=pd.DataFrame(task_k_selection)
task_k_eval_perq_df=pd.DataFrame(task_k_eval_perq)

task_k_summary_rows=[];task_k_ndcg_tests=[]
for ds,g in task_k_eval_perq_df.groupby('dataset',sort=True):
    delta=g.ndcg10_delta_selected_minus_fixed60.to_numpy(float)
    mean,lo,hi=bootstrap_mean_ci(delta,seed=STAT_SEED+30000+sum(map(ord,ds)))
    _,p=wilcoxon_safe(delta)
    task_k_ndcg_tests.append({'dataset':ds,'mean_ndcg_delta':mean,'ci_low':lo,'ci_high':hi,'p_raw':p,'rank_biserial':paired_rank_biserial(delta)})
    task_k_summary_rows.append({
        'dataset':ds,'selected_k':float(g.selected_k.iloc[0]),'n_heldout':len(g),
        'selected_cert_rate':float(g.selected_certified.mean()),'fixed60_cert_rate':float(g.fixed60_certified.mean()),
        'cert_rate_gain':float(g.selected_certified.mean()-g.fixed60_certified.mean()),
        'selected_mean_reads':float(g.selected_reads.mean()),'fixed60_mean_reads':float(g.fixed60_reads.mean()),
        'selected_mean_saving':float(g.selected_saving.mean()),'fixed60_mean_saving':float(g.fixed60_saving.mean()),
        'selected_mean_prefix_len':float(g.selected_prefix_len.mean()),'fixed60_mean_prefix_len':float(g.fixed60_prefix_len.mean()),
        'selected_ndcg10':float(g.selected_ndcg10.mean()),'fixed60_ndcg10':float(g.fixed60_ndcg10.mean()),
        'ndcg10_delta':float(delta.mean()),
    })

task_k_summary_df=pd.DataFrame(task_k_summary_rows)
task_k_ndcg_tests_df=pd.DataFrame(task_k_ndcg_tests)
task_k_ndcg_tests_df['p_holm_4']=holm_adjust(task_k_ndcg_tests_df.p_raw.to_numpy(float))
task_k_ndcg_tests_df['significant_holm_0.05']=task_k_ndcg_tests_df.p_holm_4<0.05

task_k_calibration_df.to_csv(CANON_TABLES/'secondary_task_k_calibration.csv',index=False)
task_k_selection_df.to_csv(CANON_TABLES/'secondary_task_k_selection.csv',index=False)
task_k_eval_perq_df.to_csv(CANON_TABLES/'secondary_task_k_heldout_per_query.csv',index=False)
task_k_summary_df.to_csv(CANON_TABLES/'secondary_task_k_heldout_summary.csv',index=False)
task_k_ndcg_tests_df.to_csv(CANON_STATS/'secondary_task_k_heldout_ndcg_holm4.csv',index=False)

display(task_k_calibration_df)
display(task_k_selection_df)
display(task_k_summary_df)
display(task_k_ndcg_tests_df)


## 24. Secondary post-audit real-BM25 absolute-drift analysis

The original controlled real-BM25 inference is retained unchanged: signed nDCG effects and the original eight-test Holm family remain the confirmatory result. Because MC-RRF is a **representation-reliability** constraint rather than a utility maximizer, this secondary analysis asks a directly aligned question: after family expansion, how far does each method move from the clean fusion in absolute nDCG@10 terms?

This endpoint was added after the canonical audit and must be described as secondary/post-audit if used in the manuscript. It is not a replacement for the original signed-effect test.


In [ ]:

real_abs_perq=family_perq_df.copy()
real_abs_perq['ordinary_abs_drift']=real_abs_perq.ordinary_delta.abs()
real_abs_perq['mc_abs_drift']=real_abs_perq.mc_delta.abs()
real_abs_perq['abs_drift_reduction']=real_abs_perq.ordinary_abs_drift-real_abs_perq.mc_abs_drift

real_abs_rows=[]
for ds,g in real_abs_perq.groupby('dataset',sort=True):
    x=g.abs_drift_reduction.to_numpy(float)
    mean,lo,hi=bootstrap_mean_ci(x,seed=STAT_SEED+40000+sum(map(ord,ds)))
    _,p=wilcoxon_safe(x)
    real_abs_rows.append({
        'dataset':ds,'n_queries':len(g),
        'ordinary_mean_abs_drift':float(g.ordinary_abs_drift.mean()),
        'mc_mean_abs_drift':float(g.mc_abs_drift.mean()),
        'mean_abs_drift_reduction':float(mean),'ci_low':float(lo),'ci_high':float(hi),
        'p_raw':float(p),'rank_biserial':paired_rank_biserial(x),
    })
real_abs_df=pd.DataFrame(real_abs_rows)
real_abs_df['p_holm_3']=holm_adjust(real_abs_df.p_raw.to_numpy(float))
real_abs_df['significant_holm_0.05']=real_abs_df.p_holm_3<0.05
real_abs_perq.to_csv(CANON_TABLES/'secondary_real_bm25_family_absolute_drift_per_query.csv',index=False)
real_abs_df.to_csv(CANON_STATS/'secondary_real_bm25_family_absolute_drift_holm3.csv',index=False)
display(real_abs_df)


## 25. Canonical integrity report and package

Protocol and implementation invariants still fail loudly. Scientific outcomes, however, are never used to destroy the result package: if a directional or significance pattern changes, the notebook records a warning and still writes the canonical report and ZIP so the manuscript can be updated honestly.

In [ ]:
# Structural/protocol guards: these are true integrity requirements.
assert len(primary_df2)==8, f'Expected 8 primary scaling comparisons, got {len(primary_df2)}'
assert len(primary_test_df)==8
assert len(pert_stats)==24, f'Expected 24 mechanism tests, got {len(pert_stats)}'
assert len(kk_grid_df)==12, f'Expected 12 k x K cells, got {len(kk_grid_df)}'
assert len(policy_stats_df)==16, f'Expected 16 StableRRF policy tests, got {len(policy_stats_df)}'
assert len(prov_macro2)==5, f'Expected 5 provenance split points, got {len(prov_macro2)}'
assert set(EXPECTED_Q)==set(DS_CANON)
assert (exact_df.exact_collapse_order_preservation==1.0).all()
assert (exact_df.mc_order_preservation==1.0).all()

# Scientific outcomes are never used as crash conditions. Report them honestly.
scientific_flags={
    'primary_lower_drift': int((primary_df2.drift_reduction>0).sum()),
    'primary_higher_set': int((primary_df2.set_preservation_gain>0).sum()),
    'primary_holm_significant': int(primary_df2['significant_0.05'].sum()),
    'perturb_lower_drift': int((pert_stats.drift_reduction>0).sum()),
    'perturb_higher_set': int((pert_stats.set_gain>0).sum()),
    'perturb_holm_significant': int(pert_stats['significant_0.05'].sum()),
    'kk_all8_lower_drift_cells': int((kk_grid_df.conditions_mc_lower_abs_drift==8).sum()),
    'kk_all8_higher_set_cells': int((kk_grid_df.conditions_mc_higher_set_preservation==8).sum()),
    'overgroup_increases_drift': int((over_df.overgroup_minus_correct_abs_drift>0).sum()),
}
for key,val in scientific_flags.items():
    print(key,':',val)

warnings_out=[]
if scientific_flags['primary_lower_drift']<8:warnings_out.append('Primary MC lower-drift result is not 8/8.')
if scientific_flags['primary_higher_set']<8:warnings_out.append('Primary MC set-preservation result is not 8/8.')
if scientific_flags['primary_holm_significant']<8:warnings_out.append('Not all 8 primary drift reductions are Holm-significant.')
if scientific_flags['perturb_lower_drift']<24:warnings_out.append('Perturbation lower-drift result is not 24/24.')
if scientific_flags['perturb_higher_set']<24:warnings_out.append('Perturbation set-preservation result is not 24/24.')
if scientific_flags['perturb_holm_significant']<24:warnings_out.append('Not all 24 perturbation drift reductions are Holm-significant.')
if scientific_flags['kk_all8_lower_drift_cells']<12:warnings_out.append('Not all 12 k x K cells favor MC on drift in all 8 conditions.')
if scientific_flags['kk_all8_higher_set_cells']<12:warnings_out.append('Not all 12 k x K cells favor MC on set preservation in all 8 conditions.')
if warnings_out:
    print('\nSCIENTIFIC OUTCOME WARNINGS (package will still be written):')
    for w in warnings_out:print('-',w)

lines=['# InvariantRRF canonical strict four-dataset rerun','',f'- Strict model SHA-256: `{model_hash}`','- Dense checkpoint: fixed epoch 3, no validation/test checkpoint selection, strict=True reload.','- Dense source for all four datasets: the same locked checkpoint; historical `run_dense.json` files are not reused.','']

lines += ['## Input preflight']
for r in preflight_df.itertuples():
    lines.append(f'- {r.dataset}: queries={r.queries}; documents={r.documents}; dense generated depth={r.dense_generated_depth}; primary dense cap={r.dense_primary_cap}; SPLADE={bool(r.splade)}')

lines += ['','## Exact-copy diagnostic (depth 50, one added exact copy)']
for r in exact_dataset.itertuples():
    lines.append(f'- {r.dataset}: ordinary order change={r.ordinary_order_change:.4f}; exact-collapse preservation={r.exact_collapse_order_preservation:.4f}; MC preservation={r.mc_order_preservation:.4f}')

lines += ['','## Real BM25 family']
for r in family_df.itertuples():
    lines.append(f'- {r.dataset}: RBO={r.mean_rbo:.4f}; ordinary set={r.ordinary_set:.4f}; MC set={r.mc_set:.4f}; ordinary delta={r.ordinary_delta:+.6f}; MC delta={r.mc_delta:+.6f}')
for r in family_test_df.itertuples():
    lines.append(f'- {r.dataset} / {r.test}: mean={r.mean_difference:+.6f}; 95% CI=[{r.ci_low:+.6f},{r.ci_high:+.6f}]; Holm p={r.p_holm:.6g}')

lines += ['','## Primary source-count scaling (m=8, mild 5%, depth=100, k=60, K=10)',
          f'- MC lower absolute drift: {scientific_flags["primary_lower_drift"]}/8',
          f'- MC higher set preservation: {scientific_flags["primary_higher_set"]}/8',
          f'- Holm-significant drift reductions: {scientific_flags["primary_holm_significant"]}/8',
          f'- Macro absolute-drift reduction: {primary_df2.drift_reduction.mean():.6f}',
          f'- Macro set-preservation gain: {primary_df2.set_preservation_gain.mean():.6f}',
          f'- Ordinary set-preservation range: {primary_df2.ordinary_set_preservation.min():.4f}--{primary_df2.ordinary_set_preservation.max():.4f}',
          f'- MC set-preservation range: {primary_df2.mc_set_preservation.min():.4f}--{primary_df2.mc_set_preservation.max():.4f}']
for r in primary_df2.itertuples():
    lines.append(f'- {r.dataset} / attack {r.attacked_source}: ordinary set={r.ordinary_set_preservation:.4f}; MC set={r.mc_set_preservation:.4f}; ordinary abs drift={r.ordinary_abs_ndcg_drift:.6f}; MC abs drift={r.mc_abs_ndcg_drift:.6f}; reduction={r.drift_reduction:.6f}; Holm p={r.p_holm:.6g}')

lines += ['','## Perturbation mechanisms',
          f'- Lower drift: {scientific_flags["perturb_lower_drift"]}/24',
          f'- Higher set preservation: {scientific_flags["perturb_higher_set"]}/24',
          f'- Holm significant: {scientific_flags["perturb_holm_significant"]}/24']
for r in pert_macro.itertuples():
    lines.append(f'- {r.mechanism}: RBO={r.mean_rbo:.4f}; set gain={r.set_gain:+.6f}; drift reduction={r.drift_reduction:+.6f}')

lines += ['','## k x K',
          f'- Cells with all 8 conditions lower drift: {scientific_flags["kk_all8_lower_drift_cells"]}/12',
          f'- Cells with all 8 conditions higher set preservation: {scientific_flags["kk_all8_higher_set_cells"]}/12',
          f'- Worst macro drift reduction: {kk_grid_df.macro_abs_drift_reduction.min():.6f}']

lines += ['','## Provenance splitting']
for r in prov_macro2.itertuples():lines.append(f'- split={r.split_rate:.2f}: mass={r.attacked_mass:.6f}; set={r.set_preservation:.6f}; abs drift={r.abs_drift:.6f}')
lines.append(f'- Over-grouping increases absolute drift in {scientific_flags["overgroup_increases_drift"]}/{len(over_df)} dataset/source conditions.')

lines += ['', '## StableRRF']
for r in stable_df2.itertuples():lines.append(f'- {r.dataset} / {r.source_set} / {r.target}: cert={r.cert_rate:.4f}; reads={r.mean_candidate_reads:.2f}; saving={r.saving_pct:.2f}%')
lines += ['', '### Static depth-50 ordered certification']
for r in static_df2.itertuples():lines.append(f'- {r.dataset} / K={r.K}: {r.ordered_cert_rate:.4f}')
lines += ['', '### Scheduler audit']
for r in policy_df.itertuples():lines.append(f'- {r.dataset} / {r.policy}: cert={r.certification_rate:.4f}; mean reads={r.mean_candidate_reads:.2f}; mean saving={100*r.mean_saving:.2f}%')
for r in policy_stats_df.itertuples():lines.append(f'- {r.dataset} / {r.comparison}: mean read diff={r.mean_read_difference:+.3f}; 95% CI=[{r.ci_low:+.3f},{r.ci_high:+.3f}]; Holm p={r.p_holm_16:.6g}')
lines += ['', '### Finite-lattice oracle sample']
for r in oracle_summary.itertuples():lines.append(f'- {r.dataset}: n={r.n}; adaptive cert={r.adaptive_cert_rate:.4f}; oracle cert={r.oracle_cert_rate:.4f}; adaptive reads={r.adaptive_reads:.2f}; oracle reads={r.oracle_reads:.2f}; difference={r.adaptive_minus_oracle_reads:+.2f}')

if warnings_out:
    lines += ['','## Scientific outcome warnings'] + [f'- {w}' for w in warnings_out]
else:
    lines += ['','## Scientific outcome warnings','- None. All designated directional/count checks matched the expected pattern.']

# Secondary extension summaries. These do not redefine the primary fixed-k/cap results.
if 'deep_cap_summary' in globals():
    lines += ['','## Secondary deep-dense StableRRF sensitivity']
    for r in deep_cap_summary.itertuples():
        lines.append(f'- {r.dataset} / dense cap {int(r.dense_cap)}: order cert={r.order_cert_rate:.4f}; set cert={r.set_cert_rate:.4f}; mean reads={r.mean_candidate_reads:.2f}; saving={100*r.mean_saving:.2f}%; mean final prefix={r.mean_final_prefix_len:.2f}')
if 'task_k_summary_df' in globals():
    lines += ['','## Secondary task-level adaptive k (ranking-only calibration)']
    for r in task_k_summary_df.itertuples():
        lines.append(f'- {r.dataset}: selected k={r.selected_k:g}; held-out cert={r.selected_cert_rate:.4f} vs fixed60={r.fixed60_cert_rate:.4f}; reads={r.selected_mean_reads:.2f} vs {r.fixed60_mean_reads:.2f}; held-out nDCG delta={r.ndcg10_delta:+.6f}')
if 'real_abs_df' in globals():
    lines += ['','## Secondary post-audit real-BM25 absolute drift']
    for r in real_abs_df.itertuples():
        lines.append(f'- {r.dataset}: ordinary abs drift={r.ordinary_mean_abs_drift:.6f}; MC abs drift={r.mc_mean_abs_drift:.6f}; reduction={r.mean_abs_drift_reduction:.6f}; 95% CI=[{r.ci_low:.6f},{r.ci_high:.6f}]; exploratory Holm3 p={r.p_holm_3:.6g}')

report='\n'.join(lines)
(CANON/'CANONICAL_RESULTS_FOR_MANUSCRIPT.md').write_text(report,encoding='utf-8')
print(report)

# Write a manifest after every canonical result file except the manifest itself exists.
manifest={'model_sha256':model_hash,'protocol':canonical_protocol,'files':{}}
for p in sorted(CANON.rglob('*')):
    if p.is_file() and p.name!='RUN_MANIFEST.json':
        manifest['files'][str(p.relative_to(CANON))]=sha256_file(p)
(CANON/'RUN_MANIFEST.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')

zip_path=ROOT/'InvariantRRF_Canonical_STRICT_FourDataset_Results.zip'
with zipfile.ZipFile(zip_path,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(CANON.rglob('*')):
        if p.is_file():z.write(p,arcname=str(p.relative_to(CANON)))

print('\nUPLOAD BACK:')
print(CANON/'CANONICAL_RESULTS_FOR_MANUSCRIPT.md')
print(zip_path)


## 26. Extended upload package

The canonical baseline files and the secondary extension files are packaged together. The primary results remain identifiable by the `canonical_*` filenames; all new analyses use the `secondary_*` prefix.


In [ ]:

extended_report_lines=[
    '# InvariantRRF v4 extended audit outputs','',
    '- Primary MC-RRF and StableRRF results remain the canonical fixed-depth/fixed-k analyses.',
    '- Dense top-1000 is generated only to study StableRRF observation-cap sensitivity.',
    '- Task-level adaptive k is selected without qrels on a deterministic calibration subset and evaluated on held-out queries.',
    '- Real-family absolute drift is a secondary post-audit endpoint; the original signed-effect Holm family is retained.',
    '', '## Dense-cap StableRRF sensitivity'
]
for r in deep_cap_summary.itertuples():
    extended_report_lines.append(f'- {r.dataset} / dense cap {int(r.dense_cap)}: order cert={r.order_cert_rate:.4f}; set cert={r.set_cert_rate:.4f}; mean reads={r.mean_candidate_reads:.2f}; saving={100*r.mean_saving:.2f}%; mean prefix={r.mean_final_prefix_len:.2f}')
extended_report_lines += ['', '## Task-level adaptive k held-out results']
for r in task_k_summary_df.itertuples():
    extended_report_lines.append(f'- {r.dataset}: selected k={r.selected_k:g}; cert={r.selected_cert_rate:.4f} vs fixed60={r.fixed60_cert_rate:.4f}; reads={r.selected_mean_reads:.2f} vs {r.fixed60_mean_reads:.2f}; nDCG10 delta={r.ndcg10_delta:+.6f}')
extended_report_lines += ['', '## Real BM25 family absolute-drift secondary endpoint']
for r in real_abs_df.itertuples():
    extended_report_lines.append(f'- {r.dataset}: ordinary={r.ordinary_mean_abs_drift:.6f}; MC={r.mc_mean_abs_drift:.6f}; reduction={r.mean_abs_drift_reduction:.6f}; CI=[{r.ci_low:.6f},{r.ci_high:.6f}]; exploratory Holm3 p={r.p_holm_3:.6g}')
extended_report='\n'.join(extended_report_lines)
(CANON/'EXTENDED_RESULTS_FOR_MANUSCRIPT.md').write_text(extended_report,encoding='utf-8')
print(extended_report)

# Rebuild manifest after the extension files exist.
manifest={'model_sha256':model_hash,'protocol':canonical_protocol,'files':{}}
for p in sorted(CANON.rglob('*')):
    if p.is_file() and p.name!='RUN_MANIFEST.json':
        manifest['files'][str(p.relative_to(CANON))]=sha256_file(p)
(CANON/'RUN_MANIFEST.json').write_text(json.dumps(manifest,indent=2),encoding='utf-8')

extended_zip=ROOT/'InvariantRRF_Canonical_STRICT_DeepDense_TaskAdaptiveK_Results.zip'
with zipfile.ZipFile(extended_zip,'w',zipfile.ZIP_DEFLATED) as z:
    for p in sorted(CANON.rglob('*')):
        if p.is_file():z.write(p,arcname=str(p.relative_to(CANON)))

print('\nUPLOAD BACK:')
print(CANON/'CANONICAL_RESULTS_FOR_MANUSCRIPT.md')
print(CANON/'EXTENDED_RESULTS_FOR_MANUSCRIPT.md')
print(extended_zip)
